### Завдання 1. Переклад простих фраз англійською та іспанською
- Завдання: Створити Seq-to-Seq модель для перекладу англійських фраз на іспанські.
- Дані: Набір фраз типу: "hello" → "hola", "thank you" → "gracias".
- Ціль: Навчити модель передбачати правильну іспанську послідовність символів.
- Практика: Студенти працюють з one-hot кодуванням символів, LSTM Encoder/Decoder.

In [5]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import random

# ==============================================================================
# Крок 1: Підготуємо дані
# Створимо невеликий, але різноманітний набір фраз для навчання.
# ==============================================================================

# Візьмемо набір пар для перекладу (англійська, іспанська)
pairs = [
    ("hello", "hola"),
    ("hi", "hola"),
    ("thank you", "gracias"),
    ("thanks", "gracias"),
    ("good morning", "buenos dias"),
    ("good night", "buenas noches"),
    ("yes", "si"),
    ("no", "no"),
    ("please", "por favor"),
    ("sorry", "lo siento"),
    ("goodbye", "adios"),
    ("see you", "hasta luego"),
    ("how are you", "como estas"),
    ("what is your name", "como te llamas"),
    ("i love you", "te quiero"),
    ("congratulations", "felicidades"),
    ("welcome", "bienvenido"),
    ("excuse me", "disculpe"),
    ("i am fine", "estoy bien"),
    ("do you speak english", "hablas ingles")
]

# Перемішаємо дані, щоб модель не звикала до порядку
random.shuffle(pairs)

# ==============================================================================
# Крок 2: Створимо словники (алфавітів)
# Перетворимо кожну фразу на набір цифр
# ==============================================================================

input_texts = [p[0] for p in pairs]
target_texts = [p[1] for p in pairs]

# Напишемо спеціальні символи для декодера, щоб він знав, коли починати і закінчувати фразу
start_token = "\t"  # Символ початку
end_token = "\n"    # Символ кінця

# Зберемо унікальні символи для вхідної (англійської) та цільової (іспанської) мов
input_chars = sorted(list(set("".join(input_texts))))
# До цільових символів додамо стартовий і кінцевий токени
target_chars = sorted(list(set("".join(target_texts) + start_token + end_token)))

# Створимо словники "символ -> індекс" та "індекс -> символ" для обох мов
input_char_to_idx = {c: i for i, c in enumerate(input_chars)}
input_idx_to_char = {i: c for c, i in input_char_to_idx.items()}

target_char_to_idx = {c: i for i, c in enumerate(target_chars)}
target_idx_to_char = {i: c for c, i in target_char_to_idx.items()}

# Визначимо ключові параметри для побудови матриць
num_encoder_tokens = len(input_chars)
num_decoder_tokens = len(target_chars)
max_encoder_seq_length = max([len(txt) for txt in input_texts])
max_decoder_seq_length = max([len(txt) for txt in target_texts]) + 2  # +2 для start/end токенів

print(f"Кількість семплів: {len(pairs)}")
print(f"Кількість унікальних вхідних символів: {num_encoder_tokens}")
print(f"Кількість унікальних вихідних символів: {num_decoder_tokens}")
print(f"Максимальна довжина вхідної фрази: {max_encoder_seq_length}")
print(f"Максимальна довжина вихідної фрази: {max_decoder_seq_length}")

# ==============================================================================
# Крок 3: Проведемо векторизацію даних (One-Hot Encoding)
# Перетворимо наші фрази на числові матриці, де '1' означає наявність символу.
# ==============================================================================

num_samples = len(pairs)
# Матриця для вхідних даних енкодера
encoder_input_data = np.zeros((num_samples, max_encoder_seq_length, num_encoder_tokens), dtype="float32")
# Матриця для вхідних даних декодера (з start_token)
decoder_input_data = np.zeros((num_samples, max_decoder_seq_length, num_decoder_tokens), dtype="float32")
# Матриця для цільових даних декодера (правильна відповідь, зміщена на 1 крок)
decoder_target_data = np.zeros((num_samples, max_decoder_seq_length, num_decoder_tokens), dtype="float32")

for i, (input_text, target_text) in enumerate(pairs):
    # Заповнимо матрицю для енкодера
    for t, char in enumerate(input_text):
        encoder_input_data[i, t, input_char_to_idx[char]] = 1.0

    # Заповнимо матриці для декодера
    target_text_with_tokens = start_token + target_text + end_token
    for t, char in enumerate(target_text_with_tokens):
        decoder_input_data[i, t, target_char_to_idx[char]] = 1.0
        if t > 0:
            decoder_target_data[i, t - 1, target_char_to_idx[char]] = 1.0

# ==============================================================================
# Крок 4: Побудуємо архітектуру моделі
# Складемо наший Encoder та Decoder.
# ==============================================================================

latent_dim = 256  # Розмір прихованого стану LSTM

# --- Encoder ---
encoder_inputs = keras.Input(shape=(None, num_encoder_tokens), name="encoder_inputs")
encoder_lstm = layers.LSTM(latent_dim, return_state=True, name="encoder_lstm")
_, state_h, state_c = encoder_lstm(encoder_inputs)
encoder_states = [state_h, state_c] 

# --- Decoder ---
decoder_inputs = keras.Input(shape=(None, num_decoder_tokens), name="decoder_inputs")
decoder_lstm = layers.LSTM(latent_dim, return_sequences=True, return_state=True, name="decoder_lstm")
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
# Шар для передбачення наступного символу
decoder_dense = layers.Dense(num_decoder_tokens, activation="softmax", name="decoder_dense")
decoder_outputs = decoder_dense(decoder_outputs)

# Збиремо повну модель для навчання
model = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer="rmsprop", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

# ==============================================================================
# Крок 5: Розпочнемо навчання моделі
# Покажемо моделі наші приклади багато разів, щоб вона навчилася.
# ==============================================================================

epochs = 300
batch_size = 16

history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.2, # Використовуємо частину даних для перевірки
    verbose=1
)

# ==============================================================================
# Крок 6: Створемо модель для перекладу (Inference)
# ==============================================================================

encoder_model = keras.Model(encoder_inputs, encoder_states)

# Модель Декодера: приймає один символ і "думку", видає наступний символ і нову "думку"
decoder_state_input_h = keras.Input(shape=(latent_dim,), name="decoder_state_input_h")
decoder_state_input_c = keras.Input(shape=(latent_dim,), name="decoder_state_input_c")
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    decoder_inputs, initial_state=decoder_states_inputs
)
decoder_states_inf = [state_h_inf, state_c_inf]
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

decoder_model = keras.Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf
)

# ==============================================================================
# Крок 7: Створимо функцію для декодування послідовності
# ==============================================================================

def decode_sequence(input_seq):
    # 1. Отримаємо "думку" від енкодера
    states_value = encoder_model.predict(input_seq)
    
    # 2. Готуємо початковий символ '\t' для декодера
    target_seq = np.zeros((1, 1, num_decoder_tokens), dtype="float32")
    target_seq[0, 0, target_char_to_idx[start_token]] = 1.0

    stop_condition = False
    decoded_sentence = ""
    
    # 3. Запустемо цикл генерації
    while not stop_condition:
        # 4. Передбачимо наступний символ і отримаємо новий стан
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)
        
        # 5. Вибиремо символ з найбільшою ймовірністю
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = target_idx_to_char[sampled_token_index]
        
        # 6. Умова зупинки: якщо це кінцевий символ або досягнуто ліміту
        if sampled_char == end_token or len(decoded_sentence) > max_decoder_seq_length:
            stop_condition = True
        else:
            decoded_sentence += sampled_char
            
        # 7. Оновемо вхід для наступного кроку (передбачений символ)
        target_seq = np.zeros((1, 1, num_decoder_tokens), dtype="float32")
        target_seq[0, 0, sampled_token_index] = 1.0
        
        # 8. Оновлемо "думку"
        states_value = [h, c]

    return decoded_sentence

# ==============================================================================
# Крок 8: Створимо функцію для перекладу довільного тексту
# Об'єднаємо векторизацію і декодування для зручного використання.
# ==============================================================================

def translate_text(en_text):
    # Приведемо текст до нижнього регістру
    en_text = en_text.lower()
    
    # Створемо вхідну матрицю для енкодера
    encoder_input = np.zeros((1, max_encoder_seq_length, num_encoder_tokens), dtype="float32")
    for t, char in enumerate(en_text):
        if t >= max_encoder_seq_length:
            break
        if char in input_char_to_idx:
            encoder_input[0, t, input_char_to_idx[char]] = 1.0
            
    return decode_sequence(encoder_input)

# ==============================================================================
# Крок 9: Проведемо тестування моделі
# Перевіремо, наскільки добре модель вивчила тренувальні дані та нові фрази.
# ==============================================================================

print("\n--- Приклади перекладів з навчального набору ---")
for seq_index in range(min(10, num_samples)): # Перевіримо 10 випадкових прикладів
    input_text = input_texts[seq_index]
    
    # Готуємо вхід для функції decode_sequence
    input_seq = encoder_input_data[seq_index: seq_index + 1]
    
    decoded = decode_sequence(input_seq)
    print(f"EN: {input_text} -> ES (pred): '{decoded}' | ES (true): '{target_texts[seq_index]}'")

print("\n--- Тестування на нових фразах ---")
tests = ["hello", "thank you", "good night", "i am fine", "how are you"]
for text_to_translate in tests:
    translation = translate_text(text_to_translate)
    print(f"'{text_to_translate}' -> '{translation}'")

Кількість семплів: 20
Кількість унікальних вхідних символів: 24
Кількість унікальних вихідних символів: 24
Максимальна довжина вхідної фрази: 20
Максимальна довжина вихідної фрази: 16


Model: "functional_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)            ┃ Output Shape         ┃      Param # ┃ Connected to          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs          │ (None, None, 24)     │            0 │ -                     │
│ (InputLayer)            │                      │              │                       │
├─────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ decoder_inputs          │ (None, None, 24)     │            0 │ -                     │
│ (InputLayer)            │                      │              │                       │
├─────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ encoder_lstm (LSTM)     │ [(None, 256), (None, │      287,744 │ encoder_inputs[0][0]  │
│                         │ 256), (None, 256)]   │              │                       │
├─────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ decoder_lstm (LSTM)     │ [(None, None, 256),  │      287,744 │ decoder_inputs[0][0], │
│                         │ (None, 256), (None,  │              │ encoder_lstm[0][1],   │
│                         │ 256)]                │              │ encoder_lstm[0][2]    │
├─────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ decoder_dense (Dense)   │ (None, None, 24)     │        6,168 │ decoder_lstm[0][0]    │
└─────────────────────────┴──────────────────────┴──────────────┴───────────────────────┘

 Total params: 581,656 (2.22 MB)

 Trainable params: 581,656 (2.22 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.0469 - loss: 1.9368 - val_accuracy: 0.0469 - val_loss: 1.6331
Epoch 2/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step - accuracy: 0.0703 - loss: 1.9262 - val_accuracy: 0.0938 - val_loss: 1.6278
Epoch 3/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - accuracy: 0.0586 - loss: 1.9175 - val_accuracy: 0.0781 - val_loss: 1.6226
Epoch 4/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step - accuracy: 0.0820 - loss: 1.9089 - val_accuracy: 0.0938 - val_loss: 1.6166
Epoch 5/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - accuracy: 0.0898 - loss: 1.8993 - val_accuracy: 0.0781 - val_loss: 1.6089
Epoch 6/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - accuracy: 0.0859 - loss: 1.8875 - val_accuracy: 0.0781 - val_loss: 1.5980
Epoch 7/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step - accuracy: 0.0938 - loss: 1.8710 - val_accuracy: 0.0781 - val_loss: 1.5798
Epoch 8/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step - accuracy: 0.0938 - loss: 1.8440 - val_accuracy: 0.0938 - val_

### Завдання 2. Генерація відповідей на питання
- Завдання: Створити чат-бот, який відповідає на прості запитання.
- Дані: Пара запитання-відповідь: "What is your name?" → "I am a bot".
- Ціль: Студенти реалізують Seq-to-Seq модель для Question→Answer, тренують на невеликому наборі.
- Практика: Робота з токенізацією, генерацією символів покроково.

In [6]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import random

# ==============================================================================
# Крок 1: Підготуємо дані (набір питань та відповідей)
# ==============================================================================

pairs = [
    ("hi", "hello"),
    ("hello", "hi there!"),
    ("how are you", "i am doing great, thank you for asking"),
    ("what is your name", "i am a friendly bot"),
    ("what's your name", "you can call me a seq2seq bot"),
    ("what can you do", "i can answer some simple questions"),
    ("what do you do", "i generate answers based on questions"),
    ("who are you", "i am a bot"),
    ("who made you", "i was created by a student using tensorflow"),
    ("what is python", "python is a popular programming language"),
    ("are you a robot", "yes, i am a software robot"),
    ("bye", "goodbye!"),
    ("goodbye", "see you later!")
]

# Перемішаємо дані для кращого навчання
random.shuffle(pairs)

# ==============================================================================
# Крок 2: Почнемо створення словників символів
# ==============================================================================

questions = [p[0] for p in pairs]
answers = [p[1] for p in pairs]

# Спеціальні символи для декодера
start_token = "\t"  # Початок відповіді
end_token = "\n"    # Кінець відповіді

# Створимо алфавіти для питань та відповідей
question_chars = sorted(list(set("".join(questions))))
answer_chars = sorted(list(set("".join(answers) + start_token + end_token)))

# Створиммо словники "символ -> індекс" та "індекс -> символ"
question_char_to_idx = {c: i for i, c in enumerate(question_chars)}
question_idx_to_char = {i: c for c, i in question_char_to_idx.items()}

answer_char_to_idx = {c: i for i, c in enumerate(answer_chars)}
answer_idx_to_char = {i: c for c, i in answer_char_to_idx.items()}

# Визначимо ключові параметри
num_question_tokens = len(question_chars)
num_answer_tokens = len(answer_chars)
max_question_seq_length = max([len(txt) for txt in questions])
max_answer_seq_length = max([len(txt) for txt in answers]) + 2  # Для start/end токенів

print(f"Кількість пар 'питання-відповідь': {len(pairs)}")
print(f"Кількість унікальних символів у питаннях: {num_question_tokens}")
print(f"Кількість унікальних символів у відповідях: {num_answer_tokens}")
print(f"Максимальна довжина питання: {max_question_seq_length}")
print(f"Максимальна довжина відповіді: {max_answer_seq_length}")

# ==============================================================================
# Крок 3: Почнемо векторизацію даних (One-Hot Encoding)
# ==============================================================================

num_samples = len(pairs)
question_input_data = np.zeros((num_samples, max_question_seq_length, num_question_tokens), dtype="float32")
answer_input_data = np.zeros((num_samples, max_answer_seq_length, num_answer_tokens), dtype="float32")
answer_target_data = np.zeros((num_samples, max_answer_seq_length, num_answer_tokens), dtype="float32")

for i, (question, answer) in enumerate(pairs):
    # Векторизуємо питання для енкодера
    for t, char in enumerate(question):
        question_input_data[i, t, question_char_to_idx[char]] = 1.0

    # Векторизуємо відповідь для декодера
    answer_with_tokens = start_token + answer + end_token
    for t, char in enumerate(answer_with_tokens):
        answer_input_data[i, t, answer_char_to_idx[char]] = 1.0
        if t > 0:
            # Цільові дані (правильна відповідь) змістимо на один крок
            answer_target_data[i, t - 1, answer_char_to_idx[char]] = 1.0

# ==============================================================================
# Крок 4: Почнемо створювати архітектуру моделі (Encoder-Decoder)
# ==============================================================================

latent_dim = 256  # Розмір "думки" (прихованого стану)

# --- Encoder (що розуміє питання) ---
encoder_inputs = keras.Input(shape=(None, num_question_tokens), name="question_inputs")
encoder_lstm = layers.LSTM(latent_dim, return_state=True, name="encoder_lstm")
_, state_h, state_c = encoder_lstm(encoder_inputs)
encoder_states = [state_h, state_c]

# --- Decoder (що генерує відповідь) ---
decoder_inputs = keras.Input(shape=(None, num_answer_tokens), name="answer_inputs")
decoder_lstm = layers.LSTM(latent_dim, return_sequences=True, return_state=True, name="decoder_lstm")
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = layers.Dense(num_answer_tokens, activation="softmax", name="output_dense")
decoder_outputs = decoder_dense(decoder_outputs)

# Збиремо модель для навчання
model = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer="rmsprop", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

# ==============================================================================
# Крок 5: Проведемо навчання моделі
# ==============================================================================

epochs = 400  # Кількість епох для кращого результату
batch_size = 8

history = model.fit(
    [question_input_data, answer_input_data],
    answer_target_data,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.2,
    verbose=1
)

# ==============================================================================
# Крок 6: Почнемо створення моделей для генерації відповідей (Inference)
# ==============================================================================

# Модель енкодера: приймає питання, видає "думку"
encoder_model = keras.Model(encoder_inputs, encoder_states)

# Модель декодера: приймає символ і "думку", видає наступний символ і нову "думку"
decoder_state_input_h = keras.Input(shape=(latent_dim,))
decoder_state_input_c = keras.Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    decoder_inputs, initial_state=decoder_states_inputs
)
decoder_states_inf = [state_h_inf, state_c_inf]
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

decoder_model = keras.Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf
)

# ==============================================================================
# Крок 7: Створемо функцію для генерації відповіді
# ==============================================================================

def generate_answer(input_seq):
    # 1. Отримаємо "думку" про питання
    states_value = encoder_model.predict(input_seq, verbose=0)
    
    # 2. Починемо генерацію відповіді зі стартового символу
    target_seq = np.zeros((1, 1, num_answer_tokens), dtype="float32")
    target_seq[0, 0, answer_char_to_idx[start_token]] = 1.0

    stop_condition = False
    decoded_sentence = ""
    
    while not stop_condition:
        # 3. Передбачемо наступний символ і оновимо стан
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)
        
        # 4. Вибиремо найкращий символ
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = answer_idx_to_char[sampled_token_index]
        
        # 5. Зупинемось, якщо це кінець або відповідь задовга
        if sampled_char == end_token or len(decoded_sentence) > max_answer_seq_length:
            stop_condition = True
        else:
            decoded_sentence += sampled_char
            
        # 6. Оновимо вхід для наступного кроку
        target_seq = np.zeros((1, 1, num_answer_tokens), dtype="float32")
        target_seq[0, 0, sampled_token_index] = 1.0
        
        # 7. Оновимо "думку"
        states_value = [h, c]

    return decoded_sentence

# ==============================================================================
# Крок 8: Створимо функцію-інтерфейс для чат-бота
# ==============================================================================

def ask_bot(question_text):
    question_text = question_text.lower()
    
    # Векторизуємо вхідне питання
    input_seq = np.zeros((1, max_question_seq_length, num_question_tokens), dtype="float32")
    for t, char in enumerate(question_text):
        if t >= max_question_seq_length:
            break
        if char in question_char_to_idx:
            input_seq[0, t, question_char_to_idx[char]] = 1.0
            
    return generate_answer(input_seq)

# ==============================================================================
# Крок 9: Проведемо тестування чат-бота
# ==============================================================================

print("\n--- Перевірка на прикладах з навчального набору ---")
for i in range(len(pairs)):
    question = questions[i]
    answer = answers[i]
    
    input_seq = question_input_data[i: i + 1]
    generated = generate_answer(input_seq)
    print(f"Питання: '{question}' -> Відповідь (модель): '{generated}' | Відповідь (вірна): '{answer}'")

print("\n--- Поставте своє питання боту ---")
test_questions = ["hello", "what is your name", "who are you", "what can you do", "bye"]
for q in test_questions:
    response = ask_bot(q)
    print(f"Ви: {q}")
    print(f"Бот: {response}")

Кількість пар 'питання-відповідь': 13
Кількість унікальних символів у питаннях: 21
Кількість унікальних символів у відповідях: 28
Максимальна довжина питання: 17
Максимальна довжина відповіді: 45


Model: "functional_15"

┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)            ┃ Output Shape         ┃      Param # ┃ Connected to          ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│ question_inputs         │ (None, None, 21)     │            0 │ -                     │
│ (InputLayer)            │                      │              │                       │
├─────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ answer_inputs           │ (None, None, 28)     │            0 │ -                     │
│ (InputLayer)            │                      │              │                       │
├─────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ encoder_lstm (LSTM)     │ [(None, 256), (None, │      284,672 │ question_inputs[0][0] │
│                         │ 256), (None, 256)]   │              │                       │
├─────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ decoder_lstm (LSTM)     │ [(None, None, 256),  │      291,840 │ answer_inputs[0][0],  │
│                         │ (None, 256), (None,  │              │ encoder_lstm[0][1],   │
│                         │ 256)]                │              │ encoder_lstm[0][2]    │
├─────────────────────────┼──────────────────────┼──────────────┼───────────────────────┤
│ output_dense (Dense)    │ (None, None, 28)     │        7,196 │ decoder_lstm[0][0]    │
└─────────────────────────┴──────────────────────┴──────────────┴───────────────────────┘

 Total params: 583,708 (2.23 MB)

 Trainable params: 583,708 (2.23 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/400
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 474ms/step - accuracy: 0.0067 - loss: 1.8816 - val_accuracy: 0.0815 - val_loss: 1.7390
Epoch 2/400
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - accuracy: 0.0667 - loss: 1.8682 - val_accuracy: 0.0741 - val_loss: 1.7186
Epoch 3/400
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - accuracy: 0.0911 - loss: 1.8474 - val_accuracy: 0.0741 - val_loss: 1.6759
Epoch 4/400
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - accuracy: 0.0889 - loss: 1.8039 - val_accuracy: 0.0741 - val_loss: 1.6044
Epoch 5/400
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step - accuracy: 0.0889 - loss: 1.7581 - val_accuracy: 0.0593 - val_loss: 1.5959
Epoch 6/400
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - accuracy: 0.0800 - loss: 1.7518 - val_accuracy: 0.0815 - val_loss: 1.5514
Epoch 7/400
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - accuracy: 0.0867 - loss: 1.7253 - val_accuracy: 0.0741 - val_loss: 1.5829
Epoch 8/400
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - accuracy: 0.0889 - loss: 1.7173 - val_accuracy: 0.0741 - val_lo

### Завдання 3. Перетворення дати з англійського формату у числовий
- Завдання: Наприклад, "twenty fifth of May, 2025" → "25/05/2025".
- Дані: Набір різних дат у словесному вигляді та числовому.
- Ціль: Навчити модель перетворювати текст у структуровану форму.
- Практика: Корисно для розуміння того, як Seq-to-Seq може перетворювати формати.

In [2]:
# ==============================================================================
# 1. ПРОВЕДЕМО ІМПОРТ НЕОБХІДНИХ БІБЛІОТЕК
# ==============================================================================
import tensorflow as tf
import numpy as np
import os
import time
import re
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split

print(f"TensorFlow Version: {tf.__version__}")

# ==============================================================================
# 2. ПРОВЕДЕМО ПІДГОТОВКА ДАНИХ
# ==============================================================================
# Згенеруємо датасет для навчання.

def generate_date_data(num_examples=10000):
    """
    Генерує пари дат: текстове представлення та числовий формат.
    """
    day_words = {
        1: "first", 2: "second", 3: "third", 4: "fourth", 5: "fifth", 6: "sixth", 7: "seventh",
        8: "eighth", 9: "ninth", 10: "tenth", 11: "eleventh", 12: "twelfth", 13: "thirteenth",
        14: "fourteenth", 15: "fifteenth", 16: "sixteenth", 17: "seventeenth", 18: "eighteenth",
        19: "nineteenth", 20: "twentieth", 21: "twenty first", 22: "twenty second", 23: "twenty third",
        24: "twenty fourth", 25: "twenty fifth", 26: "twenty sixth", 27: "twenty seventh",
        28: "twenty eighth", 29: "twenty ninth", 30: "thirtieth", 31: "thirty first"
    }
    month_words = {
        1: "January", 2: "February", 3: "March", 4: "April", 5: "May", 6: "June",
        7: "July", 8: "August", 9: "September", 10: "October", 11: "November", 12: "December"
    }

    start_date = datetime(2000, 1, 1)
    end_date = datetime(2050, 12, 31)
    time_between_dates = end_date - start_date
    days_between_dates = time_between_dates.days
    
    data_pairs = []
    for _ in range(num_examples):
        random_number_of_days = np.random.randint(days_between_dates)
        random_date = start_date + timedelta(days=random_number_of_days)
        
        text_date = f"{day_words[random_date.day]} of {month_words[random_date.month]}, {random_date.year}"
        numeric_date = random_date.strftime("%d/%m/%Y")
        
        data_pairs.append((text_date, numeric_date))
        
    return zip(*data_pairs)

input_texts, target_texts = generate_date_data(20000)

print("Приклад даних:")
for i in range(3):
    print(f"Вхід: {input_texts[i]}  =>  Вихід: {target_texts[i]}")

# Створимо функцію для обробки тексту
def preprocess_sentence(s, is_target=False):
    s = s.lower().strip()
    s = re.sub(r"([,.])", r" \1", s)
    s = re.sub(r'[" "]+', " ", s)
    s = s.strip()

    if is_target:
        s = ' '.join(list(s))
        
    s = '<start> ' + s + ' <end>'
    return s

# Обробимо кожне речення з урахуванням виправлення
input_preprocessed = [preprocess_sentence(s, is_target=False) for s in input_texts]
target_preprocessed = [preprocess_sentence(s, is_target=True) for s in target_texts]

print("\nПриклад оброблених даних:")
print(f"Вхід: {input_preprocessed[0]}")
print(f"Вихід: {target_preprocessed[0]}")

# ==============================================================================
# 3. ПРОВЕДЕМО ТОКЕНІЗАЦІЮ (ПЕРЕТВОРЕННЯ ТЕКСТУ В ЧИСЛА)
# ==============================================================================

# Створимо функцію, щоб перетворити текст у послідовності чисел
def tokenize(lang):
    tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
    tokenizer.fit_on_texts(lang)
    tensor = tokenizer.texts_to_sequences(lang)
    tensor = tf.keras.preprocessing.sequence.pad_sequences(tensor, padding='post')
    return tensor, tokenizer

input_tensor, input_tokenizer = tokenize(input_preprocessed)
target_tensor, target_tokenizer = tokenize(target_preprocessed)

max_length_input = input_tensor.shape[1]
max_length_target = target_tensor.shape[1]

input_train, input_val, target_train, target_val = train_test_split(input_tensor, target_tensor, test_size=0.2)

print("\nРозміри тензорів:")
print(f"Навчальний вхідний: {input_train.shape}")
print(f"Навчальний вихідний: {target_train.shape}")


# ==============================================================================
# 4. ПРОВЕДЕМО СТВОРЕННЯ МОДЕЛІ SEQ2SEQ З МЕХАНІЗМОМ УВАГИ (ATTENTION)
# ==============================================================================
BUFFER_SIZE = len(input_train)
BATCH_SIZE = 64
steps_per_epoch = len(input_train) // BATCH_SIZE
embedding_dim = 256
units = 1024
vocab_in_size = len(input_tokenizer.word_index) + 1
vocab_tar_size = len(target_tokenizer.word_index) + 1

dataset = tf.data.Dataset.from_tensor_slices((input_train, target_train)).shuffle(BUFFER_SIZE)
dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)

class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, enc_units, batch_sz):
        super(Encoder, self).__init__()
        self.batch_sz = batch_sz
        self.enc_units = enc_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.gru = tf.keras.layers.GRU(self.enc_units, return_sequences=True, return_state=True, recurrent_initializer='glorot_uniform')

    def call(self, x, hidden):
        x = self.embedding(x)
        output, state = self.gru(x, initial_state=hidden)
        return output, state

    def initialize_hidden_state(self):
        return tf.zeros((self.batch_sz, self.enc_units))

class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, query, values):
        query_with_time_axis = tf.expand_dims(query, 1)
        score = self.V(tf.nn.tanh(self.W1(query_with_time_axis) + self.W2(values)))
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights

class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units, batch_sz):
        super(Decoder, self).__init__()
        self.batch_sz = batch_sz
        self.dec_units = dec_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.gru = tf.keras.layers.GRU(self.dec_units, return_sequences=True, return_state=True, recurrent_initializer='glorot_uniform')
        self.fc = tf.keras.layers.Dense(vocab_size)
        self.attention = BahdanauAttention(self.dec_units)

    def call(self, x, hidden, enc_output):
        context_vector, attention_weights = self.attention(hidden, enc_output)
        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)
        output, state = self.gru(x)
        output = tf.reshape(output, (-1, output.shape[2]))
        x = self.fc(output)
        return x, state, attention_weights

encoder = Encoder(vocab_in_size, embedding_dim, units, BATCH_SIZE)
decoder = Decoder(vocab_tar_size, embedding_dim, units, BATCH_SIZE)

# ==============================================================================
# 5. ПРОВЕДЕМО НАВЧАННЯ МОДЕЛІ
# ==============================================================================
optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask
    return tf.reduce_mean(loss_)

checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(optimizer=optimizer, encoder=encoder, decoder=decoder)

@tf.function
def train_step(inp, targ, enc_hidden):
    loss = 0
    with tf.GradientTape() as tape:
        enc_output, enc_hidden = encoder(inp, enc_hidden)
        dec_hidden = enc_hidden
        dec_input = tf.expand_dims([target_tokenizer.word_index['<start>']] * BATCH_SIZE, 1)

        for t in range(1, targ.shape[1]):
            predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
            loss += loss_function(targ[:, t], predictions)
            dec_input = tf.expand_dims(targ[:, t], 1)

    batch_loss = (loss / int(targ.shape[1]))
    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))
    return batch_loss

EPOCHS = 10

print("\nПочаток тренування моделі...")
for epoch in range(EPOCHS):
    start = time.time()
    enc_hidden = encoder.initialize_hidden_state()
    total_loss = 0

    for (batch, (inp, targ)) in enumerate(dataset.take(steps_per_epoch)):
        batch_loss = train_step(inp, targ, enc_hidden)
        total_loss += batch_loss

        if batch % 100 == 0:
            print(f'Епоха {epoch+1} Пакет {batch} Втрати {batch_loss.numpy():.4f}')

    if (epoch + 1) % 2 == 0:
        checkpoint.save(file_prefix=checkpoint_prefix)

    print(f'Епоха {epoch+1} Втрати {total_loss / steps_per_epoch:.4f}')
    print(f'Час на епоху: {time.time() - start:.2f} сек\n')

# ==============================================================================
# 6. ПРОВЕДЕМО ПЕРЕТВОРЕННЯ (ІНФЕРЕНС)
# ==============================================================================
def evaluate(sentence):
    sentence = preprocess_sentence(sentence, is_target=False)
    inputs = [input_tokenizer.word_index.get(i, 0) for i in sentence.split(' ')]
    inputs = tf.keras.preprocessing.sequence.pad_sequences([inputs], maxlen=max_length_input, padding='post')
    inputs = tf.convert_to_tensor(inputs)

    result = ''
    hidden = [tf.zeros((1, units))]
    enc_out, enc_hidden = encoder(inputs, hidden)
    dec_hidden = enc_hidden
    dec_input = tf.expand_dims([target_tokenizer.word_index['<start>']], 0)

    for t in range(max_length_target):
        predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_out)
        predicted_id = tf.argmax(predictions[0]).numpy()
        char = target_tokenizer.index_word.get(predicted_id, '')

        if char == '<end>':
            return result
        
        # Ми не додаємо сам токен, а пробіл, якщо він є наступним,
        # щоб потім правильно об'єднати.
        if char != '<start>':
            result += char + ' '

        dec_input = tf.expand_dims([predicted_id], 0)

    return result.strip()

def translate(sentence):
    """Обгортка для evaluate, що очищує результат."""
    result_with_spaces = evaluate(sentence)
    return result_with_spaces.replace(' ', '')

# ==============================================================================
# 7. ПРОВЕДЕМО ДЕМОНСТРАЦІЮ РЕЗУЛЬТАТІВ
# ==============================================================================
try:
    checkpoint.restore(tf.train.latest_checkpoint(checkpoint_dir))
    print("\nМодель відновлено. Демонстрація перетворення:")
    
    test_dates = [
        "twenty fifth of May, 2025",
        "third of August, 2030",
        "first of January, 2001",
        "thirtieth of November, 2045"
    ]

    for date_str in test_dates:
        translated_date = translate(date_str)
        print(f'Вхід: {date_str}')
        print(f'Прогноз: {translated_date}\n')

    print("Перевірка на новій даті:")
    new_date = "nineteenth of February, 2022"
    translated_date = translate(new_date)
    print(f'Вхід: {new_date}')
    print(f'Прогноз: {translated_date}\n')

except Exception as e:
    print(f"\nНе вдалося завантажити модель. Можливо, навчання ще не було проведено. Помилка: {e}")
    print("Пропущено етап демонстрації.")


TensorFlow Version: 2.20.0
Приклад даних:
Вхід: nineteenth of May, 2038  =>  Вихід: 19/05/2038
Вхід: twenty sixth of May, 2041  =>  Вихід: 26/05/2041
Вхід: fourteenth of June, 2005  =>  Вихід: 14/06/2005

Приклад оброблених даних:
Вхід: <start> nineteenth of may , 2038 <end>
Вихід: <start> 1 9 / 0 5 / 2 0 3 8 <end>

Розміри тензорів:
Навчальний вхідний: (16000, 8)
Навчальний вихідний: (16000, 12)

Початок тренування моделі...
Епоха 1 Пакет 0 Втрати 2.4204
Епоха 1 Пакет 100 Втрати 1.0202
Епоха 1 Пакет 200 Втрати 0.3009
Епоха 1 Втрати 0.9100
Час на епоху: 325.92 сек

Епоха 2 Пакет 0 Втрати 0.3593
Епоха 2 Пакет 100 Втрати 0.1931
Епоха 2 Пакет 200 Втрати 0.1419
Епоха 2 Втрати 0.1810
Час на епоху: 305.58 сек

Епоха 3 Пакет 0 Втрати 0.1085
Епоха 3 Пакет 100 Втрати 0.1180
Епоха 3 Пакет 200 Втрати 0.0105
Епоха 3 Втрати 0.0752
Час на епоху: 350.42 сек

Епоха 4 Пакет 0 Втрати 0.0029
Епоха 4 Пакет 100 Втрати 0.0013
Епоха 4 Пакет 200 Втрати 0.0006
Епоха 4 Втрати 0.0012
Час на епоху: 402.74 сек

Еп

### Завдання 4. Автозавершення речень
- Завдання: Модель отримує початок речення і генерує його продовження.
- Дані: Короткі речення з новин чи твітів.
- Ціль: Реалізація Seq-to-Seq як language model, що передбачає наступні символи.
- Практика: Студенти бачать покрокове передбачення символів і генерацію тексту.

In [2]:
# ==============================================================================
# 1. ПРОВЕДЕМО ІМПОРТ НЕОБХІДНИХ БІБЛІОТЕК
# ==============================================================================
import tensorflow as tf
from tensorflow import keras
import numpy as np
import re

print(f"TensorFlow Version: {tf.__version__}")

# ==============================================================================
# 2. ПРОВЕДЕМО ПІДГОТОВКУ ДАНИХ
# ==============================================================================
# Створюємо невеликий датасет з українських речень.
data = [
    "Сьогодні чудова погода.",
    "Київ - столиця України.",
    "Програмування це весело та цікаво.",
    "Нейронні мережі вивчають дані.",
    "Як твої справи?",
    "Давай підемо гуляти в парк.",
    "На столі лежить книжка.",
    "Я люблю читати наукову фантастику.",
    "Машина швидко їхала по дорозі.",
    "Ввечері зорі яскраво світять на небі.",
    "Кіт спить на м'якому килимі.",
    "Море було спокійним і тихим.",
    "Дерева шумлять від вітру.",
    "Кава допомагає прокинутись зранку.",
    "Скоро почнеться новий навчальний рік.",
]

input_texts = []
target_texts = []
input_characters = set()
target_characters = set()

for line in data:
    input_text = line
    target_text = "\t" + line + "\n" # Цільовий текст має маркери початку і кінця
    # \t - початок послідовності, \n - кінець послідовності.
    input_texts.append(input_text)
    target_texts.append(target_text)
    
    # Збирtмо всі унікальні символи для створення словника
    for char in input_text:
        if char not in input_characters:
            input_characters.add(char)
    for char in target_text:
        if char not in target_characters:
            target_characters.add(char)

input_characters = sorted(list(input_characters))
target_characters = sorted(list(target_characters))
num_encoder_tokens = len(input_characters)
num_decoder_tokens = len(target_characters)
max_encoder_seq_length = max([len(txt) for txt in input_texts])
max_decoder_seq_length = max([len(txt) for txt in target_texts])

print("Кількість речень:", len(data))
print("Кількість унікальних вхідних символів:", num_encoder_tokens)
print("Кількість унікальних вихідних символів:", num_decoder_tokens)
print("Максимальна довжина вхідного речення:", max_encoder_seq_length)
print("Максимальна довжина вихідного речення:", max_decoder_seq_length)

# Створимо словники: "символ -> індекс" та "індекс -> символ"
input_token_index = dict([(char, i) for i, char in enumerate(input_characters)])
target_token_index = dict([(char, i) for i, char in enumerate(target_characters)])
reverse_target_char_index = dict((i, char) for char, i in target_token_index.items())

# ==============================================================================
# 3. ПРОВЕДЕМО ВЕКТОРИЗАЦІЮ ДАНИХ
# ==============================================================================
# Проведемо готувати дані для подачі в модель. 
# Нам потрібно три масиви:
# 1. encoder_input_data: Вхідні речення, перетворені на послідовності індексів.
# 2. decoder_input_data: Цільові речення (з маркером \t), перетворені на індекси.
# 3. decoder_target_data: Цільові речення (з маркером \n), зміщені на один крок, 
#    у форматі one-hot encoding. Це те, що модель буде намагатись передбачити.

encoder_input_data = np.zeros(
    (len(input_texts), max_encoder_seq_length, num_encoder_tokens), dtype="float32"
)
decoder_input_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32"
)
decoder_target_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32"
)

for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    for t, char in enumerate(input_text):
        encoder_input_data[i, t, input_token_index[char]] = 1.0
    for t, char in enumerate(target_text):
        decoder_input_data[i, t, target_token_index[char]] = 1.0
        if t > 0:
            decoder_target_data[i, t - 1, target_token_index[char]] = 1.0

print("\nФорма вхідних даних для енкодера:", encoder_input_data.shape)

# ==============================================================================
# 4. СТВОРИМО МОДЕЛЬ SEQ2SEQ
# ==============================================================================
# Створимо модель ,яка складається з двох частин: енкодера та декодера.
# Енкодер читає вхідну послідовність і створює вектор стану ("думку").
# Декодер використовує цей вектор стану, щоб згенерувати вихідну послідовність.

latent_dim = 256  # Розмірність прихованого стану

# --- Енкодер ---
encoder_inputs = keras.Input(shape=(None, num_encoder_tokens))
encoder = keras.layers.LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder(encoder_inputs)
encoder_states = [state_h, state_c]

# --- Декодер ---
decoder_inputs = keras.Input(shape=(None, num_decoder_tokens))
decoder_lstm = keras.layers.LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = keras.layers.Dense(num_decoder_tokens, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

# --- ОСь повноцінна модель для навчання ---
model = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer="rmsprop", loss="categorical_crossentropy", metrics=["accuracy"]
)
model.summary()

# ==============================================================================
# 5. ПРОВЕДЕМО НАВЧАННЯ МОДЕЛІ
# ==============================================================================
batch_size = 5
epochs = 150   # епох мало ставлю, бо слабкий комп'ютер 

print("\nПочаток тренування моделі...")
model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.2, # Використовуємо частину даних для валідації
)
print("Тренування завершено.")

# ==============================================================================
# 6. ПОЧНЕМО СТВОРЕННЯ МОДЕЛЕЙ ДЛЯ ГЕНЕРАЦІЇ (INFERENCE)
# ==============================================================================

# Модель енкодера: приймає вхід і повертає вектор стану.
encoder_model = keras.Model(encoder_inputs, encoder_states)

# Модель декодера: приймає вектор стану і попередній символ,
# повертає передбачення для наступного символу і новий стан.
decoder_state_input_h = keras.Input(shape=(latent_dim,))
decoder_state_input_c = keras.Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_inputs, initial_state=decoder_states_inputs
)
decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)
decoder_model = keras.Model(
    [decoder_inputs] + decoder_states_inputs, [decoder_outputs] + decoder_states
)

# ==============================================================================
# 7. СТВОРИМО ФУНКЦІЮ ДЛЯ ГЕНЕРАЦІЇ РЕЧЕНЬ
# ==============================================================================
def decode_sequence(input_seq):
    # 1. Закодуємо вхідне речення у вектор стану.
    states_value = encoder_model.predict(input_seq)

    # 2. Створимо порожню цільову послідовність, що містить лише стартовий символ.
    target_seq = np.zeros((1, 1, num_decoder_tokens))
    target_seq[0, 0, target_token_index["\t"]] = 1.0

    stop_condition = False
    decoded_sentence = ""
    while not stop_condition:
        # 3. Подамо стан і попередній символ у декодер.
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

        # 4. Виберемо наступний символ з найбільшою ймовірністю.
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = reverse_target_char_index[sampled_token_index]
        decoded_sentence += sampled_char

        # 5. Зупинимо, якщо згенеровано символ кінця або досягнуто макс. довжини.
        if sampled_char == "\n" or len(decoded_sentence) > max_decoder_seq_length:
            stop_condition = True

        # 6. Оновимо вхід для наступного кроку декодера.
        target_seq = np.zeros((1, 1, num_decoder_tokens))
        target_seq[0, 0, sampled_token_index] = 1.0

        # 7. Оновимо стан.
        states_value = [h, c]

    return decoded_sentence

# ==============================================================================
# 8. ПРОВЕДЕМО ДЕМОНСТРАЦІЮ РЕЗУЛЬТАТІВ
# ==============================================================================
def autocomplete(prompt):
    # Векторизуємо вхідний prompt
    input_seq = np.zeros((1, max_encoder_seq_length, num_encoder_tokens), dtype="float32")
    for t, char in enumerate(prompt):
        if char in input_token_index:
             input_seq[0, t, input_token_index[char]] = 1.0
        else:
            print(f"Попередження: символ '{char}' відсутній у словнику.")

    decoded_sentence = decode_sequence(input_seq)
    print("Вхід:", prompt)
    print("Згенероване продовження:", decoded_sentence.strip())
    print("-" * 30)

print("\n--- Демонстрація автозавершення ---")
autocomplete("Сьогодні")
autocomplete("Нейронні мережі")
autocomplete("Я люблю")
autocomplete("Кіт спить на")


TensorFlow Version: 2.20.0
Кількість речень: 15
Кількість унікальних вхідних символів: 45
Кількість унікальних вихідних символів: 47
Максимальна довжина вхідного речення: 37
Максимальна довжина вихідного речення: 39

Форма вхідних даних для енкодера: (15, 37, 45)


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)    │ (None, None, 45)          │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_5 (InputLayer)    │ (None, None, 47)          │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_2 (LSTM)                 │ [(None, 256), (None,      │         309,248 │ input_layer_4[0][0]        │
│                               │ 256), (None, 256)]        │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_3 (LSTM)                 │ [(None, None, 256),       │         311,296 │ input_layer_5[0][0],       │
│                               │ (None, 256), (None, 256)] │                 │ lstm_2[0][1], lstm_2[0][2] │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_1 (Dense)               │ (None, None, 47)          │          12,079 │ lstm_3[0][0]               │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 632,623 (2.41 MB)

 Trainable params: 632,623 (2.41 MB)

 Non-trainable params: 0 (0.00 B)


Початок тренування моделі...
Epoch 1/150
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 243ms/step - accuracy: 0.0363 - loss: 2.8223 - val_accuracy: 0.0855 - val_loss: 3.2251
Epoch 2/150
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.0897 - loss: 2.7856 - val_accuracy: 0.0855 - val_loss: 3.1517
Epoch 3/150
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.0897 - loss: 2.6697 - val_accuracy: 0.0855 - val_loss: 2.9372
Epoch 4/150
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.0812 - loss: 2.5329 - val_accuracy: 0.0684 - val_loss: 2.8910
Epoch 5/150
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.0791 - loss: 2.4954 - val_accuracy: 0.0855 - val_loss: 2.8138
Epoch 6/150
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.0897 - loss: 2.4377 - val_accuracy: 0.0427 - val_loss: 2.8481
Epoch 7/150
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.0662 - loss: 2.4271 - val_accuracy: 0.0855 - val_loss: 2.8251
Epoch 8/150
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.0876 - loss: 2.4073 - v

### Завдання 5. Стиснення тексту (text summarization)
- Завдання: Створити модель, яка скорочує речення: "The cat is sitting on the mat" → "Cat on mat".
- Дані: Малі набори текстів і відповідних коротких версій.
- Ціль: Студенти навчаться Seq-to-Seq для редукції тексту, без перекладу.

In [3]:
# ==============================================================================
# 1. ПРОВЕДЕМО ІМПОРТ НЕОБХІДНИХ БІБЛІОТЕК
# ==============================================================================
import tensorflow as tf
import numpy as np
import os
import time
import re
from sklearn.model_selection import train_test_split

print(f"TensorFlow Version: {tf.__version__}")

# ==============================================================================
# 2. ПРОВЕДЕМО ПІДГОТОВКУ ДАНИХ
# ==============================================================================
# Згенеруємо датасет.
# Створимо пари "довге речення -> коротке речення", видаляючи "стоп-слова".

corpus = [
    "The cat is sitting on the mat",
    "A dog was playing in the beautiful garden",
    "The sun shines brightly in the blue sky",
    "A little girl is reading a very interesting book",
    "An old man is walking slowly with a cane",
    "The birds are singing sweet songs on the tree",
    "She is a talented artist from a small town",
    "He was a brave knight in shining armor",
    "The moon provides light in the dark night",
    "We are going to the cinema for a new movie",
    "The car is driving fast on the highway",
    "A student is studying for an important exam",
    "The flowers in the vase are very beautiful",
    "A chef is cooking a delicious meal in the kitchen",
    "The children are playing with their toys",
]

# Слова, які ми будемо видаляти
STOP_WORDS = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'in', 'on', 'at', 'with', 'by', 'for', 'of', 'to', 'very', 'sweet', 'little'}

# Створимо функцію, яка створює коротку версію речення, видаляючи стоп-слова.
def create_summary(sentence):
    words = sentence.lower().split()
    summary_words = [word for word in words if word not in STOP_WORDS]
    # Робимо перше слово з великої літери для краси
    summary = ' '.join(summary_words).capitalize()
    return summary

input_texts = corpus
target_texts = [create_summary(line) for line in corpus]

print("--- Приклади згенерованих даних ---")
for i in range(3):
    print(f"Вхід:  {input_texts[i]}")
    print(f"Вихід: {target_texts[i]}\n")
    
# ==============================================================================
# 3. ПРОВЕДЕМО ПОПЕРЕДНЮ ОБРОБКУ ТА ТОКЕНІЗАЦІЮ
# ==============================================================================

def preprocess_sentence(s):
    """Підготуємо речення: нижній регістр, токени початку/кінця."""
    s = s.lower().strip()
    # Додамо пробіли навколо знаків пунктуації, щоб вони стали окремими токенами
    s = re.sub(r"([?.!,])", r" \1 ", s)
    s = re.sub(r'[" "]+', " ", s)
    s = s.strip()
    s = '<start> ' + s + ' <end>'
    return s

input_preprocessed = [preprocess_sentence(s) for s in input_texts]
target_preprocessed = [preprocess_sentence(s) for s in target_texts]

def tokenize(lang):
    """Створимо токенізатор та перетворимо текст на послідовності чисел."""
    tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
    tokenizer.fit_on_texts(lang)
    tensor = tokenizer.texts_to_sequences(lang)
    tensor = tf.keras.preprocessing.sequence.pad_sequences(tensor, padding='post')
    return tensor, tokenizer

input_tensor, input_tokenizer = tokenize(input_preprocessed)
target_tensor, target_tokenizer = tokenize(target_preprocessed)

max_length_input = input_tensor.shape[1]
max_length_target = target_tensor.shape[1]

# Розділимо дані на навчальні та валідаційні
input_train, input_val, target_train, target_val = train_test_split(input_tensor, target_tensor, test_size=0.2)

print("\n--- Розміри тензорів ---")
print(f"Навчальний вхідний: {input_train.shape}")
print(f"Навчальний вихідний: {target_train.shape}")


# ==============================================================================
# 4. ПРОВЕДЕМО СТВОРЕННЯ МОДЕЛІ SEQ2SEQ З МЕХАНІЗМОМ УВАГИ (ATTENTION)
# ==============================================================================
# Наша архітектура моделі (Encoder, Attention, Decoder)

BUFFER_SIZE = len(input_train)
BATCH_SIZE = 5 # Малий, оскільки датасет невеликий
steps_per_epoch = len(input_train) // BATCH_SIZE
embedding_dim = 128
units = 256
vocab_in_size = len(input_tokenizer.word_index) + 1
vocab_tar_size = len(target_tokenizer.word_index) + 1

dataset = tf.data.Dataset.from_tensor_slices((input_train, target_train)).shuffle(BUFFER_SIZE)
dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)

class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, enc_units, batch_sz):
        super(Encoder, self).__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.gru = tf.keras.layers.GRU(enc_units, return_sequences=True, return_state=True, recurrent_initializer='glorot_uniform')

    def call(self, x, hidden):
        x = self.embedding(x)
        output, state = self.gru(x, initial_state=hidden)
        return output, state

    def initialize_hidden_state(self, batch_sz):
        return tf.zeros((batch_sz, units))

class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, query, values):
        query_with_time_axis = tf.expand_dims(query, 1)
        score = self.V(tf.nn.tanh(self.W1(query_with_time_axis) + self.W2(values)))
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights

class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units, batch_sz):
        super(Decoder, self).__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.gru = tf.keras.layers.GRU(dec_units, return_sequences=True, return_state=True, recurrent_initializer='glorot_uniform')
        self.fc = tf.keras.layers.Dense(vocab_size)
        self.attention = BahdanauAttention(dec_units)

    def call(self, x, hidden, enc_output):
        context_vector, _ = self.attention(hidden, enc_output)
        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)
        output, state = self.gru(x)
        output = tf.reshape(output, (-1, output.shape[2]))
        x = self.fc(output)
        return x, state

encoder = Encoder(vocab_in_size, embedding_dim, units, BATCH_SIZE)
decoder = Decoder(vocab_tar_size, embedding_dim, units, BATCH_SIZE)

# ==============================================================================
# 5. ПРОВЕДЕМО НАВЧАННЯ МОДЕЛІ
# ==============================================================================
optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask
    return tf.reduce_mean(loss_)

checkpoint_dir = './summarization_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(optimizer=optimizer, encoder=encoder, decoder=decoder)

@tf.function
def train_step(inp, targ, enc_hidden):
    loss = 0
    with tf.GradientTape() as tape:
        enc_output, enc_hidden = encoder(inp, enc_hidden)
        dec_hidden = enc_hidden
        dec_input = tf.expand_dims([target_tokenizer.word_index['<start>']] * BATCH_SIZE, 1)

        for t in range(1, targ.shape[1]):
            predictions, dec_hidden = decoder(dec_input, dec_hidden, enc_output)
            loss += loss_function(targ[:, t], predictions)
            dec_input = tf.expand_dims(targ[:, t], 1) # Teacher forcing

    batch_loss = (loss / int(targ.shape[1]))
    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))
    return batch_loss

EPOCHS = 100 
print("\n--- Початок тренування моделі ---")
for epoch in range(EPOCHS):
    start = time.time()
    enc_hidden = encoder.initialize_hidden_state(BATCH_SIZE)
    total_loss = 0

    for (batch, (inp, targ)) in enumerate(dataset.take(steps_per_epoch)):
        batch_loss = train_step(inp, targ, enc_hidden)
        total_loss += batch_loss

    if (epoch + 1) % 10 == 0:
        print(f'Епоха {epoch+1} Втрати {total_loss / steps_per_epoch:.4f}')

checkpoint.save(file_prefix=checkpoint_prefix)
print(f"Тренування завершено. Модель збережено.")


# ==============================================================================
# 6. СТВОРИМО ФУНКЦІЮ ДЛЯ СТИСНЕННЯ ТЕКСТУ (INFERENCE)
# ==============================================================================
def summarize_sentence(sentence):
    sentence = preprocess_sentence(sentence)
    inputs = [input_tokenizer.word_index.get(i, 0) for i in sentence.split(' ')]
    inputs = tf.keras.preprocessing.sequence.pad_sequences([inputs], maxlen=max_length_input, padding='post')
    inputs = tf.convert_to_tensor(inputs)

    result = ''
    hidden = encoder.initialize_hidden_state(batch_sz=1)
    enc_out, enc_hidden = encoder(inputs, hidden)
    dec_hidden = enc_hidden
    dec_input = tf.expand_dims([target_tokenizer.word_index['<start>']], 0)

    for t in range(max_length_target):
        predictions, dec_hidden = decoder(dec_input, dec_hidden, enc_out)
        predicted_id = tf.argmax(predictions[0]).numpy()
        word = target_tokenizer.index_word.get(predicted_id, '')

        if word == '<end>':
            return result.strip()
        
        if word != '<start>':
             result += word + ' '
             
        dec_input = tf.expand_dims([predicted_id], 0)

    return result.strip()


# ==============================================================================
# 7. ПРОВЕДЕМО ДЕМОНСТРАЦІЮ РЕЗУЛЬТАТІВ
# ==============================================================================
checkpoint.restore(tf.train.latest_checkpoint(checkpoint_dir))

print("\n--- Демонстрація стиснення тексту ---")

# Приклади з тренувального набору
print("--- Приклади з тренувального набору ---")
test_sentence_1 = "The cat is sitting on the mat"
print(f"Вхід:   {test_sentence_1}")
print(f"Вихід: {summarize_sentence(test_sentence_1)}\n")

test_sentence_2 = "The flowers in the vase are very beautiful"
print(f"Вхід:   {test_sentence_2}")
print(f"Вихід: {summarize_sentence(test_sentence_2)}\n")

# Нові приклади, яких модель не бачила
print("--- Нові приклади ---")
test_sentence_3 = "A boy is kicking a red ball"
print(f"Вхід:   {test_sentence_3}")
print(f"Вихід: {summarize_sentence(test_sentence_3)}\n")

test_sentence_4 = "The book is on the wooden table"
print(f"Вхід:   {test_sentence_4}")
print(f"Вихід: {summarize_sentence(test_sentence_4)}\n")


TensorFlow Version: 2.20.0
--- Приклади згенерованих даних ---
Вхід:  The cat is sitting on the mat
Вихід: Cat sitting mat

Вхід:  A dog was playing in the beautiful garden
Вихід: Dog playing beautiful garden

Вхід:  The sun shines brightly in the blue sky
Вихід: Sun shines brightly blue sky


--- Розміри тензорів ---
Навчальний вхідний: (12, 12)
Навчальний вихідний: (12, 8)

--- Початок тренування моделі ---
Епоха 10 Втрати 2.5652
Епоха 20 Втрати 2.4284
Епоха 30 Втрати 1.4928
Епоха 40 Втрати 0.7456
Епоха 50 Втрати 0.3239
Епоха 60 Втрати 0.1612
Епоха 70 Втрати 0.0751
Епоха 80 Втрати 0.0459
Епоха 90 Втрати 0.0306
Епоха 100 Втрати 0.0228
Тренування завершено. Модель збережено.

--- Демонстрація стиснення тексту ---
--- Приклади з тренувального набору ---
Вхід:   The cat is sitting on the mat
Вихід: car driving fast highway

Вхід:   The flowers in the vase are very beautiful
Вихід: sun shines brightly blue sky

--- Нові приклади ---
Вхід:   A boy is kicking a red ball
Вихід: children play

### Завдання 6. Переклад між технічними термінами
- Завдання: "CPU" → "процесор", "RAM" → "оперативна пам'ять".
- Дані: Список комп’ютерних термінів англійською та українською.
- Ціль: Показати застосування Seq-to-Seq для спеціалізованих словників, не лише для повних речень.

In [2]:
# ==============================================================================
# 1. ПРОВЕДЕМО ІМПОРТ НЕОБХІДНИХ БІБЛІОТЕК
# ==============================================================================
import tensorflow as tf
from tensorflow import keras
import numpy as np

print(f"TensorFlow Version: {tf.__version__}")

# ==============================================================================
# 2. ПРОВЕДЕМО ПІДГОТОВКУ ДАНИХ (НАШ СЛОВНИК)
# ==============================================================================
# Ствоимо наш спеціалізований словник.
term_pairs = [
    ["CPU", "центральний процесор"],
    ["GPU", "графічний процесор"],
    ["RAM", "оперативна пам'ять"],
    ["ROM", "постійна пам'ять"],
    ["SSD", "твердотільний накопичувач"],
    ["HDD", "жорсткий диск"],
    ["OS", "операційна система"],
    ["API", "інтерфейс програмування"],
    ["URL", "уніфікований локатор"],
    ["IP", "інтернет протокол"],
    ["motherboard", "материнська плата"],
    ["keyboard", "клавіатура"],
]

input_texts, target_texts = zip(*term_pairs)
target_texts_processed = ['<start> ' + text + ' <end>' for text in target_texts]

print("--- Приклади даних ---")
for i in range(3):
    print(f"'{input_texts[i]}'  ->  '{target_texts[i]}'")

# ==============================================================================
# 3. ПРОВЕДЕМО ТОКЕНІЗАЦІЮ ДАНИХ
# ==============================================================================
# Вхідні дані (англійські терміни) - токенізуємо на рівні СИМВОЛІВ.
# Вихідні дані (українські терміни) - токенізуємо на рівні СЛІВ.

# Токенізатор для вхідних символів
input_tokenizer = tf.keras.preprocessing.text.Tokenizer(char_level=True)
input_tokenizer.fit_on_texts(input_texts)
input_sequences = input_tokenizer.texts_to_sequences(input_texts)

# Токенізатор для вихідних слів
target_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='', lower=False)
target_tokenizer.fit_on_texts(target_texts_processed)
target_sequences = target_tokenizer.texts_to_sequences(target_texts_processed)

# Паддинг послідовностей
input_data = tf.keras.preprocessing.sequence.pad_sequences(input_sequences, padding='post')
target_data = tf.keras.preprocessing.sequence.pad_sequences(target_sequences, padding='post')

# Розміри словників
vocab_in_size = len(input_tokenizer.word_index) + 1
vocab_tar_size = len(target_tokenizer.word_index) + 1

# Словники для зворотнього перетворення
reverse_target_word_index = {v: k for k, v in target_tokenizer.word_index.items()}

# ==============================================================================
# 4. СТВОРИМО МОДЕЛЬ SEQ2SEQ
# ==============================================================================
embedding_dim = 128
units = 256
BATCH_SIZE = len(input_data) 

# --- Енкодер ---
encoder_inputs = keras.Input(shape=(None,))
enc_emb = keras.layers.Embedding(vocab_in_size, embedding_dim)(encoder_inputs)
encoder_lstm = keras.layers.LSTM(units, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# --- Декодер ---
decoder_inputs = keras.Input(shape=(None,))
dec_emb_layer = keras.layers.Embedding(vocab_tar_size, embedding_dim)
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = keras.layers.LSTM(units, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = keras.layers.Dense(vocab_tar_size, activation='softmax')
output = decoder_dense(decoder_outputs)

model = keras.Model([encoder_inputs, decoder_inputs], output)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

# ==============================================================================
# 5. ПРОВЕДЕМО НАВЧАННЯ МОДЕЛІ
# ==============================================================================
decoder_input_data = target_data[:, :-1]
decoder_target_data = target_data[:, 1:]

epochs = 500 

print("\n--- Початок тренування моделі ---")
model.fit([input_data, decoder_input_data], decoder_target_data,
          batch_size=BATCH_SIZE,
          epochs=epochs,
          verbose=0)
print("Тренування завершено.")

# ==============================================================================
# 6. СТВОРИМО МОДЕЛІ ДЛЯ ПЕРЕКЛАДУ (INFERENCE)
# ==============================================================================
encoder_model = keras.Model(encoder_inputs, encoder_states)

decoder_state_input_h = keras.Input(shape=(units,))
decoder_state_input_c = keras.Input(shape=(units,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
dec_emb2 = dec_emb_layer(decoder_inputs)
decoder_outputs2, state_h2, state_c2 = decoder_lstm(dec_emb2, initial_state=decoder_states_inputs)
decoder_states2 = [state_h2, state_c2]
decoder_outputs2 = decoder_dense(decoder_outputs2)
decoder_model = keras.Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2] + decoder_states2)

# ==============================================================================
# 7. СТВОРИМО ФУНКЦІЮ ДЛЯ ПЕРЕКЛАДУ ТЕРМІНІВ
# ==============================================================================
def translate_term(input_seq):
    states_value = encoder_model.predict(input_seq, verbose=0)
    
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_tokenizer.word_index['<start>']
    
    stop_condition = False
    decoded_sentence = ''
    max_len = target_data.shape[1]

    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)
        
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_target_word_index.get(sampled_token_index)
        
        if (sampled_word is None or sampled_word == '<end>' or len(decoded_sentence.split()) > max_len):
            stop_condition = True
            continue
            
        decoded_sentence += ' ' + sampled_word
        
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index
        
        states_value = [h, c]
        
    return decoded_sentence.strip()

# ==============================================================================
# 8. ПРОВЕДЕМО ДЕМОНСТРАЦІЮ РЕЗУЛЬТАТІВ
# ==============================================================================
def translate(term):
    """Функція-обгортка для зручної демонстрації."""
    input_seq = input_tokenizer.texts_to_sequences([term])
    input_tensor = tf.keras.preprocessing.sequence.pad_sequences(input_seq, maxlen=input_data.shape[1], padding='post')
    
    translation = translate_term(input_tensor)
    print(f"Англійською: '{term}'")
    print(f"Українською: '{translation}'\n")

print("\n--- Демонстрація перекладу ---")
translate('RAM')
translate('CPU')
translate('OS')
translate('keyboard')


TensorFlow Version: 2.20.0
--- Приклади даних ---
'CPU'  ->  'центральний процесор'
'GPU'  ->  'графічний процесор'
'RAM'  ->  'оперативна пам'ять'


Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)    │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_5 (InputLayer)    │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding_4 (Embedding)       │ (None, None, 128)         │           2,432 │ input_layer_4[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding_5 (Embedding)       │ (None, None, 128)         │           3,072 │ input_layer_5[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_4 (LSTM)                 │ [(None, 256), (None,      │         394,240 │ embedding_4[0][0]          │
│                               │ 256), (None, 256)]        │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_5 (LSTM)                 │ [(None, None, 256),       │         394,240 │ embedding_5[0][0],         │
│                               │ (None, 256), (None, 256)] │                 │ lstm_4[0][1], lstm_4[0][2] │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_2 (Dense)               │ (None, None, 24)          │           6,168 │ lstm_5[0][0]               │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 800,152 (3.05 MB)

 Trainable params: 800,152 (3.05 MB)

 Non-trainable params: 0 (0.00 B)


--- Початок тренування моделі ---
Тренування завершено.

--- Демонстрація перекладу ---
Англійською: 'RAM'
Українською: 'оперативна пам'ять'

Англійською: 'CPU'
Українською: 'центральний процесор'

Англійською: 'OS'
Українською: 'операційна система'

Англійською: 'keyboard'
Українською: 'клавіатура'



### Завдання 7. Перетворення чисел у текст і навпаки
- Завдання: "123" → "one hundred twenty three", "forty five" → "45".
- Дані: Малі набори чисел та текстових представлень.
- Ціль: Практика перетворень форматів за допомогою Seq-to-Seq.
- Навчання: Студенти розуміють, що Seq-to-Seq можна використовувати не лише для перекладу.

In [1]:
# ==============================================================================
# 1. ПРОВЕДЕМО ІМПОРТ НЕОБХІДНИХ БІБЛІОТЕК
# ==============================================================================
try:
    from num2words import num2words
except ImportError:
    print("Будь ласка, встановіть бібліотеку 'num2words': pip install num2words")
    exit()

import tensorflow as tf
from tensorflow import keras
import numpy as np
import re

print(f"TensorFlow Version: {tf.__version__}")

# ==============================================================================
# 2. ПРОВЕДЕМО ГЕНЕРАЦІЮ ДАНИХ
# ==============================================================================
# Створимо пари "число-текст" для чисел від 1 до 200
numbers_as_digits = [str(i) for i in range(1, 201)]
numbers_as_words = [num2words(i).replace('-', ' ') for i in range(1, 201)]

print("--- Приклади згенерованих даних ---")
for i in [0, 21, 122]:
    print(f"'{numbers_as_digits[i]}'  <->  '{numbers_as_words[i]}'")

# ==============================================================================
# ЧАСТИНА A: ПЕРЕТВОРИМО ЦИФРИ У ТЕКСТ ("123" -> "one hundred twenty three")
# ==============================================================================
print("\n" + "="*60)
print("ЧАСТИНА A: НАВЧАННЯ МОДЕЛІ 'ЦИФРИ -> ТЕКСТ'")
print("="*60)

# ------------------------------------------------------------------------------
# A.1. Підготуємо дані та проведемо токенізацію
# ------------------------------------------------------------------------------
input_texts_d2w = numbers_as_digits
target_texts_d2w = ['<start> ' + text + ' <end>' for text in numbers_as_words]

# Вхід (цифри) токенізуємо на рівні символів
input_tokenizer_d2w = tf.keras.preprocessing.text.Tokenizer(char_level=True)
input_tokenizer_d2w.fit_on_texts(input_texts_d2w)
input_seq_d2w = input_tokenizer_d2w.texts_to_sequences(input_texts_d2w)

# Вихід (слова) токенізуємо на рівні слів
target_tokenizer_d2w = tf.keras.preprocessing.text.Tokenizer(filters='')
target_tokenizer_d2w.fit_on_texts(target_texts_d2w)
target_seq_d2w = target_tokenizer_d2w.texts_to_sequences(target_texts_d2w)

# Паддинг
input_data_d2w = tf.keras.preprocessing.sequence.pad_sequences(input_seq_d2w, padding='post')
target_data_d2w = tf.keras.preprocessing.sequence.pad_sequences(target_seq_d2w, padding='post')

vocab_in_size_d2w = len(input_tokenizer_d2w.word_index) + 1
vocab_tar_size_d2w = len(target_tokenizer_d2w.word_index) + 1
reverse_target_word_index_d2w = {v: k for k, v in target_tokenizer_d2w.word_index.items()}

# ------------------------------------------------------------------------------
# A.2. Створимо та навчимо моделі
# ------------------------------------------------------------------------------
embedding_dim = 64
units = 128

# Модель (аналогічна попереднім завданням)
encoder_inputs = keras.Input(shape=(None,), name='encoder_input_d2w')
enc_emb = keras.layers.Embedding(vocab_in_size_d2w, embedding_dim)(encoder_inputs)
encoder_lstm = keras.layers.LSTM(units, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

decoder_inputs = keras.Input(shape=(None,), name='decoder_input_d2w')
dec_emb_layer = keras.layers.Embedding(vocab_tar_size_d2w, embedding_dim)
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = keras.layers.LSTM(units, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = keras.layers.Dense(vocab_tar_size_d2w, activation='softmax')
output = decoder_dense(decoder_outputs)

model_d2w = keras.Model([encoder_inputs, decoder_inputs], output)
model_d2w.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

decoder_input_data = target_data_d2w[:, :-1]
decoder_target_data = target_data_d2w[:, 1:]

print("\nНавчання моделі 'цифри -> текст'...")
model_d2w.fit([input_data_d2w, decoder_input_data], decoder_target_data,
            batch_size=32, epochs=200, verbose=0)
print("Навчання завершено.")

# ------------------------------------------------------------------------------
# A.3. Inference та демонстрація
# ------------------------------------------------------------------------------
encoder_model_d2w = keras.Model(encoder_inputs, encoder_states)
decoder_state_input_h = keras.Input(shape=(units,))
decoder_state_input_c = keras.Input(shape=(units,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
dec_emb2 = dec_emb_layer(decoder_inputs)
decoder_outputs2, state_h2, state_c2 = decoder_lstm(dec_emb2, initial_state=decoder_states_inputs)
decoder_states2 = [state_h2, state_c2]
decoder_outputs2 = decoder_dense(decoder_outputs2)
decoder_model_d2w = keras.Model([decoder_inputs] + decoder_states_inputs, [decoder_outputs2] + decoder_states2)

def convert_digits_to_words(input_seq):
    states_value = encoder_model_d2w.predict(input_seq, verbose=0)
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_tokenizer_d2w.word_index['<start>']
    stop_condition = False
    decoded_sentence = ''
    while not stop_condition:
        output_tokens, h, c = decoder_model_d2w.predict([target_seq] + states_value, verbose=0)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_target_word_index_d2w.get(sampled_token_index)
        if (sampled_word is None or sampled_word == '<end>' or len(decoded_sentence.split()) > 20):
            stop_condition = True
            continue
        decoded_sentence += ' ' + sampled_word
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index
        states_value = [h, c]
    return decoded_sentence.strip()

def test_d2w(digits_str):
    input_seq = input_tokenizer_d2w.texts_to_sequences([digits_str])
    input_tensor = tf.keras.preprocessing.sequence.pad_sequences(input_seq, maxlen=input_data_d2w.shape[1], padding='post')
    result = convert_digits_to_words(input_tensor)
    print(f"Вхід: '{digits_str}' -> Результат: '{result}'")

print("\n--- Демонстрація 'цифри -> текст' ---")
test_d2w('42')
test_d2w('150')
test_d2w('199')

# ==============================================================================
# ЧАСТИНА Б: ПЕРЕТВОРИМО ТЕКСТ В ЦИФРИ ("one hundred..." -> "123")
# ==============================================================================
print("\n" + "="*60)
print("ЧАСТИНА Б: НАВЧАННЯ МОДЕЛІ 'ТЕКСТ -> ЦИФРИ'")
print("="*60)

# ------------------------------------------------------------------------------
# Б.1. Підготуємо дані та проведемо токенізацію
# ------------------------------------------------------------------------------
input_texts_w2d = numbers_as_words
# Цільові дані (цифри) готуємо для char-level токенізації, додаючи пробіли
target_texts_w2d = ['<start> ' + ' '.join(list(text)) + ' <end>' for text in numbers_as_digits]

# Вхід (слова) токенізуємо на рівні слів
input_tokenizer_w2d = tf.keras.preprocessing.text.Tokenizer(filters='')
input_tokenizer_w2d.fit_on_texts(input_texts_w2d)
input_seq_w2d = input_tokenizer_w2d.texts_to_sequences(input_texts_w2d)

# Вихід (цифри) токенізуємо як "слова" (окремі символи)
target_tokenizer_w2d = tf.keras.preprocessing.text.Tokenizer(filters='')
target_tokenizer_w2d.fit_on_texts(target_texts_w2d)
target_seq_w2d = target_tokenizer_w2d.texts_to_sequences(target_texts_w2d)

# Паддинг
input_data_w2d = tf.keras.preprocessing.sequence.pad_sequences(input_seq_w2d, padding='post')
target_data_w2d = tf.keras.preprocessing.sequence.pad_sequences(target_seq_w2d, padding='post')

vocab_in_size_w2d = len(input_tokenizer_w2d.word_index) + 1
vocab_tar_size_w2d = len(target_tokenizer_w2d.word_index) + 1
reverse_target_char_index_w2d = {v: k for k, v in target_tokenizer_w2d.word_index.items()}

# ------------------------------------------------------------------------------
# Б.2. Створимо та навчимо моделі
# ------------------------------------------------------------------------------
encoder_inputs_w2d = keras.Input(shape=(None,), name='encoder_input_w2d')
enc_emb_w2d = keras.layers.Embedding(vocab_in_size_w2d, embedding_dim)(encoder_inputs_w2d)
encoder_lstm_w2d = keras.layers.LSTM(units, return_state=True)
_, state_h_w2d, state_c_w2d = encoder_lstm_w2d(enc_emb_w2d)
encoder_states_w2d = [state_h_w2d, state_c_w2d]

decoder_inputs_w2d = keras.Input(shape=(None,), name='decoder_input_w2d')
dec_emb_layer_w2d = keras.layers.Embedding(vocab_tar_size_w2d, embedding_dim)
dec_emb_w2d = dec_emb_layer_w2d(decoder_inputs_w2d)
decoder_lstm_w2d = keras.layers.LSTM(units, return_sequences=True, return_state=True)
decoder_outputs_w2d, _, _ = decoder_lstm_w2d(dec_emb_w2d, initial_state=encoder_states_w2d)
decoder_dense_w2d = keras.layers.Dense(vocab_tar_size_w2d, activation='softmax')
output_w2d = decoder_dense_w2d(decoder_outputs_w2d)

model_w2d = keras.Model([encoder_inputs_w2d, decoder_inputs_w2d], output_w2d)
model_w2d.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

decoder_input_data_w2d = target_data_w2d[:, :-1]
decoder_target_data_w2d = target_data_w2d[:, 1:]

print("\nНавчання моделі 'текст -> цифри'...")
model_w2d.fit([input_data_w2d, decoder_input_data_w2d], decoder_target_data_w2d,
            batch_size=32, epochs=200, verbose=0)
print("Навчання завершено.")

# ------------------------------------------------------------------------------
# Б.3. Inference та демонстрація
# ------------------------------------------------------------------------------
encoder_model_w2d = keras.Model(encoder_inputs_w2d, encoder_states_w2d)
decoder_state_input_h_w2d = keras.Input(shape=(units,))
decoder_state_input_c_w2d = keras.Input(shape=(units,))
decoder_states_inputs_w2d = [decoder_state_input_h_w2d, decoder_state_input_c_w2d]
dec_emb2_w2d = dec_emb_layer_w2d(decoder_inputs_w2d)
decoder_outputs2_w2d, state_h2_w2d, state_c2_w2d = decoder_lstm_w2d(dec_emb2_w2d, initial_state=decoder_states_inputs_w2d)
decoder_states2_w2d = [state_h2_w2d, state_c2_w2d]
decoder_outputs2_w2d = decoder_dense_w2d(decoder_outputs2_w2d)
decoder_model_w2d = keras.Model([decoder_inputs_w2d] + decoder_states_inputs_w2d, [decoder_outputs2_w2d] + decoder_states2_w2d)

def convert_words_to_digits(input_seq):
    states_value = encoder_model_w2d.predict(input_seq, verbose=0)
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_tokenizer_w2d.word_index['<start>']
    stop_condition = False
    decoded_digits = ''
    while not stop_condition:
        output_tokens, h, c = decoder_model_w2d.predict([target_seq] + states_value, verbose=0)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = reverse_target_char_index_w2d.get(sampled_token_index)
        if (sampled_char is None or sampled_char == '<end>' or len(decoded_digits) > 5):
            stop_condition = True
            continue
        decoded_digits += sampled_char
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index
        states_value = [h, c]
    return decoded_digits

def test_w2d(words_str):
    input_seq = input_tokenizer_w2d.texts_to_sequences([words_str])
    input_tensor = tf.keras.preprocessing.sequence.pad_sequences(input_seq, maxlen=input_data_w2d.shape[1], padding='post')
    result = convert_words_to_digits(input_tensor)
    print(f"Вхід: '{words_str}' -> Результат: '{result}'")

print("\n--- Демонстрація 'текст -> цифри' ---")
test_w2d('seventy seven')
test_w2d('one hundred fifty two')
test_w2d('thirty three')


TensorFlow Version: 2.20.0
--- Приклади згенерованих даних ---
'1'  <->  'one'
'22'  <->  'twenty two'
'123'  <->  'one hundred and twenty three'

ЧАСТИНА A: НАВЧАННЯ МОДЕЛІ 'ЦИФРИ -> ТЕКСТ'

Навчання моделі 'цифри -> текст'...
Навчання завершено.

--- Демонстрація 'цифри -> текст' ---
Вхід: '42' -> Результат: 'forty two'
Вхід: '150' -> Результат: 'one hundred and fifty'
Вхід: '199' -> Результат: 'one hundred and ninety nine'

ЧАСТИНА Б: НАВЧАННЯ МОДЕЛІ 'ТЕКСТ -> ЦИФРИ'

Навчання моделі 'текст -> цифри'...
Навчання завершено.

--- Демонстрація 'текст -> цифри' ---
Вхід: 'seventy seven' -> Результат: '77'
Вхід: 'one hundred fifty two' -> Результат: '150'
Вхід: 'thirty three' -> Результат: '33'


### Завдання 8. Переклад емоційних твітів у спрощений формат
- Завдання: Модель отримує твіт зі смайлами або сленгом: "I love this " → "I like this".
- Дані: Набори твітів та їх спрощених версій.
- Ціль: Використання Seq-to-Seq для нормалізації тексту.

In [7]:
# ==============================================================================
# 1. ПРОВЕДЕМО ІМПОРТ НЕОБХІДНИХ БІБЛІОТЕК
# ==============================================================================
import tensorflow as tf
from tensorflow import keras
import numpy as np
import re

print(f"TensorFlow Version: {tf.__version__}")

# ==============================================================================
# 2. ПРОВЕДЕМО ПІДГОТОВКУ ДАНИХ
# ==============================================================================
# Створюємо наш набір даних.
tweet_pairs = [
    ["i love this so much <3", "i like this very much"],
    ["omg this is awesome!!!", "this is great"],
    ["cant wait for the weekend :D", "i am looking forward to the weekend"],
    ["lol u r so funny", "you are very funny"],
    ["im so tired zzz", "i am very tired"],
    ["thx for the help", "thank you for the help"],
    ["idk what to do :/", "i do not know what to do"],
    ["c u later", "see you later"],
    ["this is the best day everrrrr", "this is a very good day"],
    ["im bored af", "i am very bored"],
    ["wanna grab lunch?", "do you want to get lunch?"],
    ["gr8 job!", "great job"]
]

input_texts, target_texts = zip(*tweet_pairs)

# ==============================================================================
# 3. ПРОВЕДЕМО ПОПЕРЕДНЮ ОБРОБКУ ТА ТОКЕНІЗАЦІЮ
# ==============================================================================

def preprocess_sentence(s):
    """Підготуємо речення: видалимо зайві символи, додамо токени.
       Залишио лише літери, цифри та основні смайли/символи """
    s = s.lower()
    s = re.sub(r'([?.!,])', r' \1 ', s)
    s = re.sub(r'[" "]+', " ", s)
    s = s.strip()
    return s

def preprocess_target_sentence(s):
    """Підготуємо цільове речення з токенами start/end."""
    s = preprocess_sentence(s)
    s = '<start> ' + s + ' <end>'
    return s

input_preprocessed = [preprocess_sentence(s) for s in input_texts]
target_preprocessed = [preprocess_target_sentence(s) for s in target_texts]

print("--- Приклади оброблених даних ---")
print(f"Вхід: {input_preprocessed[0]}")
print(f"Ціль: {target_preprocessed[0]}\n")

# Створимо токенізатори для обох "мов"
# Використовуємо filters='', щоб не видаляти < >
input_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
input_tokenizer.fit_on_texts(input_preprocessed)
input_sequences = input_tokenizer.texts_to_sequences(input_preprocessed)

target_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
target_tokenizer.fit_on_texts(target_preprocessed)
target_sequences = target_tokenizer.texts_to_sequences(target_preprocessed)

# Паддинг
input_data = tf.keras.preprocessing.sequence.pad_sequences(input_sequences, padding='post')
target_data = tf.keras.preprocessing.sequence.pad_sequences(target_sequences, padding='post')

vocab_in_size = len(input_tokenizer.word_index) + 1
vocab_tar_size = len(target_tokenizer.word_index) + 1

# Словники для зворотнього перетворення
reverse_input_word_index = {v: k for k, v in input_tokenizer.word_index.items()}
reverse_target_word_index = {v: k for k, v in target_tokenizer.word_index.items()}

# ==============================================================================
# 4. ПРОВЕДЕМО СТВОРЕННЯ МОДЕЛІ SEQ2SEQ
# ==============================================================================
embedding_dim = 128
units = 256
BATCH_SIZE = len(input_data)

# --- Енкодер ---
encoder_inputs = keras.Input(shape=(None,))
enc_emb = keras.layers.Embedding(vocab_in_size, embedding_dim)(encoder_inputs)
encoder_lstm = keras.layers.LSTM(units, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# --- Декодер ---
decoder_inputs = keras.Input(shape=(None,))
dec_emb_layer = keras.layers.Embedding(vocab_tar_size, embedding_dim)
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = keras.layers.LSTM(units, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = keras.layers.Dense(vocab_tar_size, activation='softmax')
output = decoder_dense(decoder_outputs)

model = keras.Model([encoder_inputs, decoder_inputs], output)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

# ==============================================================================
# 5. ПРОВЕДЕМО НАВЧАННЯ МОДЕЛІ
# ==============================================================================
decoder_input_data = target_data[:, :-1]
decoder_target_data = target_data[:, 1:]

epochs = 300

print("\n--- Початок тренування моделі ---")
history = model.fit([input_data, decoder_input_data], decoder_target_data,
          batch_size=BATCH_SIZE,
          epochs=epochs,
          verbose=0)
print("Тренування завершено.")

# ==============================================================================
# 6. ПРОВЕДЕМО СТВОРЕННЯ МОДЕЛЕЙ ДЛЯ НОРМАЛІЗАЦІЇ (INFERENCE)
# ==============================================================================
encoder_model = keras.Model(encoder_inputs, encoder_states)

decoder_state_input_h = keras.Input(shape=(units,))
decoder_state_input_c = keras.Input(shape=(units,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
dec_emb2 = dec_emb_layer(decoder_inputs)
decoder_outputs2, state_h2, state_c2 = decoder_lstm(dec_emb2, initial_state=decoder_states_inputs)
decoder_states2 = [state_h2, state_c2]
decoder_outputs2 = decoder_dense(decoder_outputs2)
decoder_model = keras.Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2] + decoder_states2)

# ==============================================================================
# 7. СТВОРИМО ФУНКЦІЮ ДЛЯ НОРМАЛІЗАЦІЇ
# ==============================================================================
def normalize_tweet(input_seq):
    states_value = encoder_model.predict(input_seq, verbose=0)
    
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_tokenizer.word_index['<start>']
    
    stop_condition = False
    decoded_sentence = ''
    max_len = target_data.shape[1]

    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)
        
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_target_word_index.get(sampled_token_index)
        
        if (sampled_word is None or sampled_word == '<end>' or len(decoded_sentence.split()) > max_len):
            stop_condition = True
            continue
            
        decoded_sentence += ' ' + sampled_word
        
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index
        
        states_value = [h, c]
        
    return decoded_sentence.strip()

# ==============================================================================
# 8. ПРОВЕДЕМО ДЕМОНСТРАЦІЮ РЕЗУЛЬТАТІВ
# ==============================================================================
def normalize(tweet):
    """Функція-обгортка для зручної демонстрації."""
    preprocessed = preprocess_sentence(tweet)
    input_seq = input_tokenizer.texts_to_sequences([preprocessed])
    input_tensor = tf.keras.preprocessing.sequence.pad_sequences(input_seq, maxlen=input_data.shape[1], padding='post')
    
    normalized_text = normalize_tweet(input_tensor)
    print(f"Оригінальний твіт: '{tweet}'")
    print(f"Нормалізовано:   '{normalized_text}'\n")

print("\n--- Демонстрація нормалізації ---")
normalize("lol that was great")
normalize("im so sleepy zzz")
normalize("thx so much <3")


TensorFlow Version: 2.20.0
--- Приклади оброблених даних ---
Вхід: i love this so much <3
Ціль: <start> i like this very much <end>



Model: "functional_15"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_20 (InputLayer)   │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_21 (InputLayer)   │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding_8 (Embedding)       │ (None, None, 128)         │           5,632 │ input_layer_20[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding_9 (Embedding)       │ (None, None, 128)         │           4,864 │ input_layer_21[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_10 (LSTM)                │ [(None, 256), (None,      │         394,240 │ embedding_8[0][0]          │
│                               │ 256), (None, 256)]        │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_11 (LSTM)                │ [(None, None, 256),       │         394,240 │ embedding_9[0][0],         │
│                               │ (None, 256), (None, 256)] │                 │ lstm_10[0][1],             │
│                               │                           │                 │ lstm_10[0][2]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_9 (Dense)               │ (None, None, 38)          │           9,766 │ lstm_11[0][0]              │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 808,742 (3.09 MB)

 Trainable params: 808,742 (3.09 MB)

 Non-trainable params: 0 (0.00 B)


--- Початок тренування моделі ---
Тренування завершено.

--- Демонстрація нормалізації ---
Оригінальний твіт: 'lol that was great'
Нормалізовано:   'see you later'

Оригінальний твіт: 'im so sleepy zzz'
Нормалізовано:   'i am very bored'

Оригінальний твіт: 'thx so much <3'
Нормалізовано:   'thank you for the help'



### Завдання 9. Автоматична транскрипція абревіатур
- Завдання: "ASAP" → "as soon as possible", "BRB" → "be right back".
- Дані: Малий словник абревіатур і розшифровок.
- Ціль: Практика Seq-to-Seq для розпізнавання та розшифровки скорочень.

In [6]:
# ==============================================================================
# 1. ПРОВЕДЕМО ІМПОРТ НЕОБХІДНИХ БІБЛІОТЕК
# ==============================================================================
import tensorflow as tf
from tensorflow import keras
import numpy as np
from sklearn.model_selection import train_test_split

print(f"TensorFlow Version: {tf.__version__}")

# ==============================================================================
# 2. ПРОВЕДЕМО ПІДГОТОВКУ ДАНИХ (СЛОВНИК АБРЕВІАТУР)
# ==============================================================================
# Створюємо наш словник.
abbreviation_dict = {
    'ASAP': 'as soon as possible',
    'BRB': 'be right back',
    'LOL': 'laughing out loud',
    'IMHO': 'in my humble opinion',
    'BTW': 'by the way',
    'FYI': 'for your information',
    'IDK': 'i dont know', # "don't" спрощено для легкості токенізації
    'OMG': 'oh my god',
    'TTYL': 'talk to you later',
    'NP': 'no problem'
}

input_texts = list(abbreviation_dict.keys())
target_texts = list(abbreviation_dict.values())

# Додамо токени початку та кінця до цільових фраз
target_texts_processed = ['<start> ' + text + ' <end>' for text in target_texts]

print("--- Приклади даних ---")
for i in range(3):
    print(f"{input_texts[i]}  ->  {target_texts[i]}")

# ==============================================================================
# 3. ПРОВЕДЕМО ТОКЕНІЗАЦІЮ ДАНИХ
# ==============================================================================
# Вхідні дані (абревіатури) - токенізуємо на рівні СИМВОЛІВ.
# Вихідні дані (фрази) - токенізуємо на рівні СЛІВ.

# Токенізатор для вхідних символів
input_tokenizer = tf.keras.preprocessing.text.Tokenizer(char_level=True)
input_tokenizer.fit_on_texts(input_texts)
input_sequences = input_tokenizer.texts_to_sequences(input_texts)

# Токенізатор для вихідних слів
target_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
target_tokenizer.fit_on_texts(target_texts_processed)
target_sequences = target_tokenizer.texts_to_sequences(target_texts_processed)

# Паддинг послідовностей до однакової довжини
input_data = tf.keras.preprocessing.sequence.pad_sequences(input_sequences, padding='post')
target_data = tf.keras.preprocessing.sequence.pad_sequences(target_sequences, padding='post')

# Розміри словників
vocab_in_size = len(input_tokenizer.word_index) + 1
vocab_tar_size = len(target_tokenizer.word_index) + 1

print(f"\nКількість унікальних символів на вході: {vocab_in_size}")
print(f"Кількість унікальних слів на виході: {vocab_tar_size}")

# Словники для зворотнього перетворення
reverse_input_char_index = {v: k for k, v in input_tokenizer.word_index.items()}
reverse_target_word_index = {v: k for k, v in target_tokenizer.word_index.items()}

# ==============================================================================
# 4. ПРОВЕДЕМО СТВОРЕННЯ МОДЕЛІ SEQ2SEQ
# ==============================================================================
embedding_dim = 64
units = 128
BATCH_SIZE = len(input_data) 

# --- Енкодер ---
encoder_inputs = keras.Input(shape=(None,))
enc_emb = keras.layers.Embedding(vocab_in_size, embedding_dim)(encoder_inputs)
encoder_lstm = keras.layers.LSTM(units, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

# --- Декодер ---
decoder_inputs = keras.Input(shape=(None,))
dec_emb_layer = keras.layers.Embedding(vocab_tar_size, embedding_dim)
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = keras.layers.LSTM(units, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = keras.layers.Dense(vocab_tar_size, activation='softmax')
output = decoder_dense(decoder_outputs)

model = keras.Model([encoder_inputs, decoder_inputs], output)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

# ==============================================================================
# 5. ПРОВЕДЕМО НАВЧАННЯ МОДЕЛІ
# ==============================================================================
# Готуємо дані для навчання: вхід для декодера і ціль для декодера
decoder_input_data = target_data[:, :-1]
decoder_target_data = target_data[:, 1:]

epochs = 200

print("\n--- Початок тренування моделі ---")
history = model.fit([input_data, decoder_input_data], decoder_target_data,
          batch_size=BATCH_SIZE,
          epochs=epochs,
          verbose=0) # Вимкнемо логування епох, щоб не засмічувати вивід
print("Тренування завершено.")

# ==============================================================================
# 6. ПРОВЕДЕМО СТВОРЕННЯ МОДЕЛЕЙ ДЛЯ РОЗШИФРОВКИ (INFERENCE)
# ==============================================================================
encoder_model = keras.Model(encoder_inputs, encoder_states)

decoder_state_input_h = keras.Input(shape=(units,))
decoder_state_input_c = keras.Input(shape=(units,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
dec_emb2 = dec_emb_layer(decoder_inputs)
decoder_outputs2, state_h2, state_c2 = decoder_lstm(dec_emb2, initial_state=decoder_states_inputs)
decoder_states2 = [state_h2, state_c2]
decoder_outputs2 = decoder_dense(decoder_outputs2)
decoder_model = keras.Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2] + decoder_states2)

# ==============================================================================
# 7. СТВОРИМО ФУНКЦІЮ ДЛЯ РОЗШИФРОВКИ АБРЕВІАТУР
# ==============================================================================
def transcribe_abbreviation(input_seq):
    states_value = encoder_model.predict(input_seq)
    
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_tokenizer.word_index['<start>']
    
    stop_condition = False
    decoded_sentence = ''
    max_len = target_data.shape[1]

    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)
        
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_target_word_index.get(sampled_token_index)
        
        if (sampled_word is None or sampled_word == '<end>' or len(decoded_sentence.split()) > max_len):
            stop_condition = True
            continue
            
        decoded_sentence += ' ' + sampled_word
        
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index
        
        states_value = [h, c]
        
    return decoded_sentence.strip()

# ==============================================================================
# 8.ПРОВЕДЕМО ДЕМОНСТРАЦІЮ РЕЗУЛЬТАТІВ
# ==============================================================================
def transcribe(abbreviation):
    """Функція-обгортка для зручної демонстрації."""
    input_seq = input_tokenizer.texts_to_sequences([abbreviation])
    input_tensor = tf.keras.preprocessing.sequence.pad_sequences(input_seq, maxlen=input_data.shape[1], padding='post')
    
    transcription = transcribe_abbreviation(input_tensor)
    print(f"Абревіатура: '{abbreviation}'")
    print(f"Розшифровка: '{transcription}'\n")

print("\n--- Демонстрація розшифровки ---")
transcribe('ASAP')
transcribe('LOL')
transcribe('IDK')
transcribe('NP')


TensorFlow Version: 2.20.0
--- Приклади даних ---
ASAP  ->  as soon as possible
BRB  ->  be right back
LOL  ->  laughing out loud

Кількість унікальних символів на вході: 19
Кількість унікальних слів на виході: 33


Model: "functional_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_16 (InputLayer)   │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_17 (InputLayer)   │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding_6 (Embedding)       │ (None, None, 64)          │           1,216 │ input_layer_16[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding_7 (Embedding)       │ (None, None, 64)          │           2,112 │ input_layer_17[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_8 (LSTM)                 │ [(None, 128), (None,      │          98,816 │ embedding_6[0][0]          │
│                               │ 128), (None, 128)]        │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_9 (LSTM)                 │ [(None, None, 128),       │          98,816 │ embedding_7[0][0],         │
│                               │ (None, 128), (None, 128)] │                 │ lstm_8[0][1], lstm_8[0][2] │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_8 (Dense)               │ (None, None, 33)          │           4,257 │ lstm_9[0][0]               │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 205,217 (801.63 KB)

 Trainable params: 205,217 (801.63 KB)

 Non-trainable params: 0 (0.00 B)


--- Початок тренування моделі ---
Тренування завершено.

--- Демонстрація розшифровки ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step
Абревіатура: 'ASAP'
Розшифровка: 'as soon as possible'

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
Абревіатура: 'LOL'
Розшифровка: 'laughing out loud'

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Абревіатура: 'IDK'
Розшифровка: 'i dont know'

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
Абревіатура: 'NP'
Розшифровка: 'no problem'



### Завдання 10. Створення простого перекладача між вигаданими мовами
- Завдання: Студенти створюють штучний словник, наприклад: "abc" → "xyz", "hello" → "qwer".
- Дані: Створюють власні набори “словосполучень” для навчання.
- Ціль: Показати повний цикл: генерація даних, навчання, тестування на нових реченнях.
- Практика: Підкреслює, що Seq-to-Seq працює для будь-яких послідовностей.

In [4]:
# ==============================================================================
# 1. ПРОВЕДЕМО ІМПОРТ НЕОБХІДНИХ БІБЛІОТЕК
# ==============================================================================
import tensorflow as tf
from tensorflow import keras
import numpy as np
import re
from sklearn.model_selection import train_test_split

print(f"TensorFlow Version: {tf.__version__}")

# ==============================================================================
# 2. ПРОВЕДЕМО ГЕНЕРАЦІЮ ДАНИХ (СТВОРЕННЯ МОВИ ТА СЛОВНИКА)
# ==============================================================================
# Мова 1: "Людська" (прості англійські слова)
# Мова 2: "Ельфійська" (вигадані слова)

# Створимо словник для перекладу
human_to_elf_dict = {
    'hello': 'aven',
    'world': 'telum',
    'cat': 'cael',
    'dog': 'niram',
    'sits': 'lim',
    'on': 'na',
    'mat': 'sor',
    'the': 'i',
    'a': 'il',
    'sun': 'anor',
    'shines': 'cal',
    'moon': 'sil',
    'is': 'na',
    'bright': 'laure',
    'dark': 'mor',
    'tree': 'orn',
}

human_vocab = list(human_to_elf_dict.keys())

def generate_sentence_pairs(num_pairs):
    """Генеруємо пари речень на "людській" та "ельфійській" мовах."""
    pairs = []
    for _ in range(num_pairs):
        # Складемо речення з 3-5 випадкових слів
        sentence_len = np.random.randint(3, 6)
        human_sentence_words = np.random.choice(human_vocab, sentence_len, replace=False)
        
        human_sentence = " ".join(human_sentence_words)
        # Перекладемо слово за словом, зберігаючи порядок
        elf_sentence = " ".join([human_to_elf_dict[word] for word in human_sentence_words])
        
        pairs.append([human_sentence, elf_sentence])
    return pairs

# Генеруємо 2000 речень для навчання
data = generate_sentence_pairs(2000)
input_texts, target_texts = zip(*data)

print("--- Приклади згенерованих даних ---")
for i in range(3):
    print(f"Людська: {input_texts[i]}")
    print(f"Ельфійська: {target_texts[i]}\n")
    
# ==============================================================================
# 3. ПРОВЕДЕМО ПОПЕРЕДНЮ ОБРОБКУ ТА ТОКЕНІЗАЦІЮ
# ==============================================================================

def preprocess_sentence(s):
    """Підготуємо речення: додамо токенів початку/кінця."""
    return '<start> ' + s.lower() + ' <end>'

input_preprocessed = [preprocess_sentence(s) for s in input_texts]
target_preprocessed = [preprocess_sentence(s) for s in target_texts]

def tokenize(lang):
    """Створимо токенізатор та перетворимо текст на послідовності чисел."""
    tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
    tokenizer.fit_on_texts(lang)
    tensor = tokenizer.texts_to_sequences(lang)
    tensor = tf.keras.preprocessing.sequence.pad_sequences(tensor, padding='post')
    return tensor, tokenizer

# Проведемо токенізацію на рівні слів
input_tensor, input_tokenizer = tokenize(input_preprocessed)
target_tensor, target_tokenizer = tokenize(target_preprocessed)

# Створимо словники для перетворення індексів назад у слова
reverse_input_word_index = {v: k for k, v in input_tokenizer.word_index.items()}
reverse_target_word_index = {v: k for k, v in target_tokenizer.word_index.items()}

# Розділимо дані
input_train, input_val, target_train, target_val = train_test_split(input_tensor, target_tensor, test_size=0.2)

print(f"Розмір вхідного словника: {len(input_tokenizer.word_index)+1}")
print(f"Розмір вихідного словника: {len(target_tokenizer.word_index)+1}")
print(f"Форма вхідного тензора: {input_train.shape}")
print(f"Форма вихідного тензора: {target_train.shape}")

# ==============================================================================
# 4. СТВОРИМО МОДЕЛЬ SEQ2SEQ
# ==============================================================================
# Використаємо просту архітектуру Енкодер-Декодер з LSTM шарами.

embedding_dim = 128
units = 256
vocab_in_size = len(input_tokenizer.word_index) + 1
vocab_tar_size = len(target_tokenizer.word_index) + 1
BUFFER_SIZE = len(input_train)
BATCH_SIZE = 64
steps_per_epoch = len(input_train) // BATCH_SIZE

dataset = tf.data.Dataset.from_tensor_slices((input_train, target_train)).shuffle(BUFFER_SIZE)
dataset = dataset.batch(BATCH_SIZE, drop_remainder=True)

# --- Енкодер ---
encoder_inputs = keras.Input(shape=(None,))
enc_emb = keras.layers.Embedding(vocab_in_size, embedding_dim)(encoder_inputs)
encoder_lstm = keras.layers.LSTM(units, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c] # Контекстний вектор

# --- Декодер ---
decoder_inputs = keras.Input(shape=(None,))
dec_emb_layer = keras.layers.Embedding(vocab_tar_size, embedding_dim)
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = keras.layers.LSTM(units, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = keras.layers.Dense(vocab_tar_size, activation='softmax')
output = decoder_dense(decoder_outputs)

# --- Повна модель для навчання ---
model = keras.Model([encoder_inputs, decoder_inputs], output)
model.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy')
model.summary()

# ==============================================================================
# 5. ПРОВЕДЕМО НАВЧАННЯ МОДЕЛІ
# ==============================================================================
# Готуємо дані для декодера. 
# Вхід для декодера - це цільове речення, зсунуте на один крок вправо. 
# Вихід - це оригінальне цільове речення.
decoder_input_data_train = target_train[:, :-1]
decoder_target_data_train = target_train[:, 1:]
decoder_input_data_val = target_val[:, :-1]
decoder_target_data_val = target_val[:, 1:]

epochs = 30

print("\n--- Початок тренування моделі ---")
history = model.fit([input_train, decoder_input_data_train], decoder_target_data_train,
          batch_size=BATCH_SIZE,
          epochs=epochs,
          validation_data=([input_val, decoder_input_data_val], decoder_target_data_val))
print("Тренування завершено.")

# ==============================================================================
# 6. ПРОВЕДЕМО СТВОРЕННЯ МОДЕЛЕЙ ДЛЯ ПЕРЕКЛАДУ (INFERENCE)
# ==============================================================================
# Модель енкодера
encoder_model = keras.Model(encoder_inputs, encoder_states)

# Модель декодера
decoder_state_input_h = keras.Input(shape=(units,))
decoder_state_input_c = keras.Input(shape=(units,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
dec_emb2 = dec_emb_layer(decoder_inputs)
decoder_outputs2, state_h2, state_c2 = decoder_lstm(dec_emb2, initial_state=decoder_states_inputs)
decoder_states2 = [state_h2, state_c2]
decoder_outputs2 = decoder_dense(decoder_outputs2)
decoder_model = keras.Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs2] + decoder_states2)

# ==============================================================================
# 7. СТВОРИМО ФУНКЦІЮ ДЛЯ ПЕРЕКЛАДУ РЕЧЕНЬ
# ==============================================================================
def translate_sentence(input_seq):
    states_value = encoder_model.predict(input_seq)
    
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_tokenizer.word_index['<start>']
    
    stop_condition = False
    decoded_sentence = ''
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)
        
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = reverse_target_word_index.get(sampled_token_index)
        
        if (sampled_word == '<end>' or len(decoded_sentence.split()) > input_seq.shape[1]):
            stop_condition = True
            continue
            
        decoded_sentence += ' ' + sampled_word
        
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index
        
        states_value = [h, c]
        
    return decoded_sentence.strip()

# ==============================================================================
# 8. ПРОВЕДЕМО ДЕМОНСТРАЦІЮ РЕЗУЛЬТАТІВ
# ==============================================================================
def translate(sentence):
    """Створимо функцію-обгортку для зручної демонстрації."""
    preprocessed = preprocess_sentence(sentence)
    tokens = preprocessed.split(' ')
    input_seq = [input_tokenizer.word_index.get(word, 0) for word in tokens]
    input_tensor = keras.preprocessing.sequence.pad_sequences([input_seq], maxlen=input_train.shape[1], padding='post')
    
    translation = translate_sentence(input_tensor)
    print(f"Людська: '{sentence}'")
    print(f"Переклад: '{translation}'\n")

print("\n--- Демонстрація перекладача ---")

# Перекладемо декілька речень з валідаційного набору
for i in range(5):
    input_seq = input_val[i:i+1]
    original_sentence = ' '.join([reverse_input_word_index.get(idx, '?') for idx in input_seq[0] if idx > 2])
    translation = translate_sentence(input_seq)
    print(f"Оригінал: '{original_sentence}'")
    print(f"Переклад: '{translation}'\n")

# Перекладемо нове речення, якого модель не бачила
print("--- Тестування на новому реченні ---")
translate("hello tree and sun")


TensorFlow Version: 2.20.0
--- Приклади згенерованих даних ---
Людська: shines sits bright sun
Ельфійська: cal lim laure anor

Людська: tree a moon cat
Ельфійська: orn il sil cael

Людська: shines sun mat bright tree
Ельфійська: cal anor sor laure orn

Розмір вхідного словника: 19
Розмір вихідного словника: 18
Форма вхідного тензора: (1600, 7)
Форма вихідного тензора: (1600, 7)


Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8 (InputLayer)    │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_9 (InputLayer)    │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding_2 (Embedding)       │ (None, None, 128)         │           2,432 │ input_layer_8[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding_3 (Embedding)       │ (None, None, 128)         │           2,304 │ input_layer_9[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_4 (LSTM)                 │ [(None, 256), (None,      │         394,240 │ embedding_2[0][0]          │
│                               │ 256), (None, 256)]        │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_5 (LSTM)                 │ [(None, None, 256),       │         394,240 │ embedding_3[0][0],         │
│                               │ (None, 256), (None, 256)] │                 │ lstm_4[0][1], lstm_4[0][2] │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_6 (Dense)               │ (None, None, 18)          │           4,626 │ lstm_5[0][0]               │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 797,842 (3.04 MB)

 Trainable params: 797,842 (3.04 MB)

 Non-trainable params: 0 (0.00 B)


--- Початок тренування моделі ---
Epoch 1/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - loss: 2.4802 - val_loss: 2.1308
Epoch 2/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 2.0693 - val_loss: 1.9417
Epoch 3/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 1.9479 - val_loss: 1.8976
Epoch 4/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 1.8898 - val_loss: 1.9208
Epoch 5/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 1.8458 - val_loss: 1.8759
Epoch 6/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 1.8065 - val_loss: 1.7220
Epoch 7/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 1.7307 - val_loss: 1.6290
Epoch 8/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 1.5494 - val_loss: 1.7518
Epoch 9/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 1.4405 - val_loss: 1.3117
Epoch 10/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 1.2562 - val_loss: 1.1197
Epoch 11/30
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 1.0199 - val_loss: 0.9183
Epoch 12/30
25/25 ━━━━━━━━━━